# FabricOps Guardrail Validation

**Run in Microsoft Fabric Spark, with the installed FabricOps Starter Kit.** This disposable notebook owns all fixtures, frozen Data Contracts, tests, results and cleanup. No external datasets, companion modules or authoring widgets are required.

The detailed matrix calls the public `check_*()` functions first. They still resolve real frozen Data Contracts and persist `METADATA_GUARDRAIL_RESULTS`. A smaller suite then verifies real Read/Write orchestration, stage stopping, Guardrail Coverage and publication.

**Setup:** attach a Fabric Environment with the package installed; configure the three existing Development Lakehouses in section 00 (or reuse standard `FABRIC_CONTEXT`). Metadata must already be provisioned by standard setup. Run 00–08 top to bottom. Section 09 is inert until you explicitly enter the printed namespace after reviewing results.

**Validation record:** Microsoft Fabric execution has **not** been performed. Local schema, syntax, fixture definitions and assertion-helper checks do not establish Fabric runtime success. Production defects must remain FAIL with diagnostics; this notebook never patches production functions or fabricates results.

**Coverage promise:** the finite matrix below covers named supported options, Warn/Block actions and relevant clean/boundary/null/empty/failure branches. It does not claim every possible input, regex or PySpark expression. `TEST_FILTER` can narrow an investigation; every excluded case is reported SKIP, never PASS.


## 00. Environment

Standard FabricOps configuration; Development only. Fill in Lakehouse coordinates below only when no standard context is already active. Shared metadata tables are verified, not provisioned here.


In [ ]:
import builtins
import io
import json
import re
import traceback
import uuid
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from collections import Counter
import math
from datetime import datetime, timedelta

from pyspark.sql import SparkSession, functions as F
from delta.tables import DeltaTable
from fabricops_kit import (
    __version__, FabricStore, FrameworkConfig, PathConfig, setup_notebook,
    orchestrate_read, orchestrate_write, pipeline_read, pipeline_write, profile_table, resolve_table_id,
    check_dq, check_schema, check_freshness, check_source_drift,
    check_sensitive_data, check_guardrail_coverage,
    read_lakehouse_table, write_lakehouse_table,
)
from fabricops_kit.config.shared import get_default_fabric_context, get_store
from fabricops_kit.config.audit import build_runtime_audit_fields
from fabricops_kit.config.metadata_schemas import (
    metadata_table_physical_schema, metadata_table_schema_registry,
)
from fabricops_kit.io.shared import resolve_configured_lakehouse_table
from fabricops_kit.pipeline.shared import add_target_audit_fields, resolve_target_audit_fields
# Existing UI-independent authoritative services; deliberately internal maintainer usage.
from fabricops_kit.data_contract.shared import (
    create_contract_draft, save_contract_draft, freeze_contract,
    canonical_guardrail_rule_record,
)


In [ ]:
# Standalone bootstrap: supply YOUR DEVELOPMENT coordinates only. Never commit real IDs.
DEVELOPMENT_STORES = {
    "Bronze": {"workspace_id": "", "item_id": "", "schema_enabled": True, "schema": "dbo"},
    "Silver": {"workspace_id": "", "item_id": "", "schema_enabled": True, "schema": "dbo"},
    "Metadata": {"workspace_id": "", "item_id": "", "schema_enabled": True, "schema": "governance"},
}
try:
    ORIGINAL_CONTEXT = get_default_fabric_context()
except RuntimeError:
    ORIGINAL_CONTEXT = None
if ORIGINAL_CONTEXT is None:
    missing = [key for key, value in DEVELOPMENT_STORES.items()
               if not value["workspace_id"] or not value["item_id"]]
    if missing:
        raise RuntimeError("Configure Development Lakehouse coordinates in DEVELOPMENT_STORES: " + ", ".join(missing))
    CONFIG = FrameworkConfig(path_config=PathConfig(paths={
        "dev": {key: FabricStore(env="dev", kind="lakehouse", **value)
                for key, value in DEVELOPMENT_STORES.items()}
    }))
    ENV = "dev"
else:
    CONFIG, ENV = ORIGINAL_CONTEXT["config"], ORIGINAL_CONTEXT["env"]
if ENV != "dev":
    raise RuntimeError(f"Development-only notebook: expected env='dev', found {ENV!r}.")
spark = SparkSession.getActiveSession()
if spark is None:
    raise RuntimeError("An active Microsoft Fabric Spark session is required.")
spark.conf.set("spark.sql.session.timeZone", CONFIG.audit_timezone)
STORES = {key: get_store(CONFIG, ENV, key) for key in ("Bronze", "Silver", "Metadata")}


In [ ]:
for key, store in STORES.items():
    if store.env != "dev" or store.kind != "lakehouse":
        raise RuntimeError(f"{key} must be a configured Development Lakehouse.")
    # Refuse a Development data store that aliases a configured Production data store.
    if key != "Metadata":
        for env_key, env_stores in CONFIG.path_config.paths.items():
            if str(env_key).lower() in {"prod", "production"}:
                for prod in env_stores.values():
                    if (store.workspace_id, store.item_id) == (prod.workspace_id, prod.item_id):
                        raise RuntimeError(f"{key} aliases a configured Production item; refusing mutations.")
for key, store in STORES.items():
    hroot = spark._jvm.org.apache.hadoop.fs.Path(store.root + "/Tables")
    try:
        assert hroot.getFileSystem(spark._jsc.hadoopConfiguration()).exists(hroot)
    except Exception as exc:
        raise RuntimeError(f"Required Development Lakehouse unavailable: {key}") from exc
RUN_CONTEXT = setup_notebook(CONFIG, env=ENV, required_targets=list(STORES))
# Standalone maintainer notebooks are outside the numbered template naming policy.
# Retain that diagnostic and exempt only naming; all environment failures still stop execution.
failed_checks = [c for c in RUN_CONTEXT.validation_results if c.status == "fail" and c.name != "notebook_naming"]
if failed_checks:
    raise RuntimeError(f"Fabric prerequisites unavailable: {failed_checks}")
print("Standard setup checks:", RUN_CONTEXT.validation_results)
BASE_CONTEXT = dict(ORIGINAL_CONTEXT or {
    "config": CONFIG, "env": ENV, "runtime_metadata": RUN_CONTEXT.runtime_metadata,
})
FABRIC_CONTEXT = dict(BASE_CONTEXT)
builtins.FABRIC_CONTEXT = FABRIC_CONTEXT
# Fail before seeding when audit identity or routed metadata is unavailable.
build_runtime_audit_fields(config=CONFIG, env=ENV, runtime_context=FABRIC_CONTEXT)
META_NAMES = (
    "METADATA_DATA_CATALOGUE", "METADATA_DATA_PROFILED", "METADATA_DATA_PROFILED_FREQUENCY",
    "METADATA_DATA_LINEAGE", "METADATA_DATA_CONTRACT", "METADATA_ENRICHMENT",
    "METADATA_GUARDRAIL", "METADATA_GUARDRAIL_RESULTS", "METADATA_SOURCE_OBSERVATION",
)
def metadata(name):
    """Read a canonical Metadata table through the configured Metadata Lakehouse."""
    return read_lakehouse_table(name, store="Metadata",
        schema=metadata_table_physical_schema(CONFIG, name), spark_session=spark,
        context=FABRIC_CONTEXT)
for name in META_NAMES:
    try:
        actual = set(metadata(name).columns)
        required = set(metadata_table_schema_registry()[name].fieldNames())
        assert required <= actual, f"missing columns: {sorted(required - actual)}"
    except Exception as exc:
        raise RuntimeError(f"Required configured Metadata table unavailable/incompatible: {name}") from exc

# Do not reset this manifest by rerunning section 00 in the same session.
if globals().get("OWNED") and not globals().get("CLEANED", False):
    raise RuntimeError("This session already owns a test run. Review/clean it before starting another.")
RUN_NAMESPACE = "fo_gr_ut_" + uuid.uuid4().hex[:16]
OWNED, CONTRACTS, RESULTS, ARTIFACTS, TASKS = {}, {}, {}, {}, {}
COLUMN_DOCS = {}
TEST_FILTER = ""  # Empty runs everything; otherwise a substring of a Test ID.
INCLUDE_PUBLICATION_TESTS = True  # False explicitly reports publication-dependent cases SKIP.
CLEANED = False
# Fixed records/timestamps; broad clean freshness window keeps fixtures reproducible across years.
FIXED_TIME = datetime(2025, 1, 15, 12, 0, 0)
CLEAN_MAX_AGE_DAYS = 36500
print(f"FabricOps {__version__}; Fabric runtime validation pending; namespace={RUN_NAMESPACE}")
print("Cleanup confirmation value:", RUN_NAMESPACE)


## 01. Seed test data

Four stable synthetic records; `row_id` is the test identity and `business_key` is independently mutable. `make_frame()` always starts from the clean records. Most check tests only mutate an in-memory DataFrame. Freshness/Source Drift use real test-owned physical sources.

Every reservation loads its own canonical column identities immediately, including dynamically created targets. Publication targets are separate from the check target. Their physical fixtures already include package-generated audit columns, while their initial Catalogue snapshot describes the business input; all valid publication contracts are frozen before publication.


In [ ]:
FIXTURE_SCHEMA = "row_id int, business_key string, name string, category string, amount double, ceiling double, code string, required_note string, email string, age int, secret string, modified_at timestamp, partition_day date"
BASE_ROWS = [
    (1, "K1", "Alice", "A", 20.0, 100.0, "AA01", "ready", "alpha@example.test", 25, "private-one", FIXED_TIME, FIXED_TIME.date()),
    (2, "K2", "Bob", "B", 40.0, 100.0, "BB02", "ready", "beta@example.test", 45, "private-two", FIXED_TIME, FIXED_TIME.date()),
    (3, "K3", "Cara", "A", 60.0, 100.0, "CC03", "ready", "alpha@example.test", 65, "private-three", FIXED_TIME, (FIXED_TIME + timedelta(days=1)).date()),
    (4, "K4", "Dan", "B", 80.0, 100.0, "DD04", "ready", "gamma@example.test", 85, "private-four", FIXED_TIME, (FIXED_TIME + timedelta(days=1)).date()),
]
FIELD_NAMES = [part.strip().split()[0] for part in FIXTURE_SCHEMA.split(",")]

def fixture_records(changes=None, keep_ids=None):
    """Apply explicit per-row replacements; no check implementation is reproduced."""
    rows = [dict(zip(FIELD_NAMES, row)) for row in BASE_ROWS]
    for row in rows:
        row.update((changes or {}).get(row["row_id"], {}))
    return [row for row in rows if keep_ids is None or row["row_id"] in keep_ids]

def make_frame(changes=None, keep_ids=None):
    """Build a typed Spark fixture from clean records and explicit replacements."""
    rows = fixture_records(changes, keep_ids)
    return spark.createDataFrame(rows, FIXTURE_SCHEMA)

def clean_frame():
    """Return the unmodified four-record business fixture."""
    return make_frame()

def physical(role):
    """Read the exact physical table owned by a fixture role."""
    s = OWNED[role]
    return read_lakehouse_table(s["name"], store=s["store"], schema=s["schema"],
                               spark_session=spark, context=FABRIC_CONTEXT)

def put(role, frame):
    """Replace an owned fixture table without modifying other identities."""
    s = OWNED[role]
    assert ENV == "dev" and s["name"] == RUN_NAMESPACE + "_" + role
    write_lakehouse_table(frame, s["name"], store=s["store"], schema=s["schema"],
        mode="overwrite", options={"overwriteSchema": "true"}, context=FABRIC_CONTEXT, verbose=False)


In [ ]:
def reserve(role, store, publication=False):
    """Claim one previously absent identity; register and remember its business columns."""
    name = RUN_NAMESPACE + "_" + role
    schema = STORES[store].schema if STORES[store].schema_enabled else None
    table_id = resolve_table_id(store=store, schema=schema, table_name=name)
    _, _, _, path = resolve_configured_lakehouse_table(store, name, schema, context=FABRIC_CONTEXT)
    hpath = spark._jvm.org.apache.hadoop.fs.Path(path)
    fs = hpath.getFileSystem(spark._jsc.hadoopConfiguration())
    if fs.exists(hpath):
        raise RuntimeError(f"Physical collision for {role}; refusing to claim it.")
    for name_of_metadata in ("METADATA_DATA_CATALOGUE", "METADATA_DATA_CONTRACT"):
        if metadata(name_of_metadata).where((F.col("environment_name") == ENV) &
                (F.col("table_id") == table_id)).limit(1).count():
            raise RuntimeError(f"Metadata collision for {role}; refusing to claim it.")
    OWNED[role] = dict(name=name, store=store, schema=schema, table_id=table_id, path=path)
    data = clean_frame()
    if publication:
        data = add_target_audit_fields(data, resolve_target_audit_fields(FABRIC_CONTEXT))
        # Synthetic seed rows must not look like an already completed publication.
        data = data.withColumn("_activity_id", F.lit("fixture_seed_" + RUN_NAMESPACE))
    write_lakehouse_table(data, name, store=store, schema=schema, mode="errorifexists",
                         context=FABRIC_CONTEXT, verbose=False)
    profile_table(dataframe=clean_frame(), store=store, schema=schema, table_name=name,
                  frequency_enabled=False, spark_session=spark)
    rows = metadata("METADATA_DATA_CATALOGUE").where((F.col("environment_name") == ENV) &
                (F.col("table_id") == table_id) & F.col("is_active")).collect()
    COLUMN_DOCS[role] = [{key: row[key] for key in ("column_id", "column_name", "data_type")}
                         for row in rows if row["column_id"]]
    assert {doc["column_name"] for doc in COLUMN_DOCS[role]} == set(FIELD_NAMES)
    return OWNED[role]

SOURCE = reserve("source", "Bronze")
TARGET = reserve("checks", "Silver")
UNREAD = reserve("unread", "Bronze")
print("Owned tables:", {role: s["name"] for role, s in OWNED.items()})


## 02. Seed complete Data Contracts

All contracts use the existing authoritative draft → save → freeze services. No frozen definition is edited. Rules retain the authoring service's action fields; the notebook does not insert compensating fields for runtime bugs.

The cached contract key includes the role, action, full rule parameters and processing definition. All test contracts use the immutable initial business-column snapshot for that owned role. Later profiling cannot add operational columns to future test expectations.


In [ ]:
def rule(kind, rule_type, parameters=None, column=None):
    """Describe one authored Guardrail definition and its parameters."""
    return dict(kind=kind, rule_type=rule_type, parameters=parameters or {}, column=column)

SCHEMA = rule("schema", "strict")
FRESH = rule("freshness", "max_age", {
    "freshness_column": "modified_at", "maximum_age": CLEAN_MAX_AGE_DAYS, "maximum_age_unit": "days"})
DRIFT = rule("source_drift", "historical_mutation", {
    "partition_column": "partition_day", "change_column": "modified_at"})

@contextmanager
def selection(specs, overrides=None):
    """Select exact frozen versions temporarily and restore the prior overrides."""
    old = FABRIC_CONTEXT.get("data_contract_overrides")
    selected = {s["table_id"]: {"contract_id": s["contract_id"], "contract_version": s["version"]}
                for s in specs}
    selected.update(overrides or {})
    FABRIC_CONTEXT["data_contract_overrides"] = selected
    try:
        yield
    finally:
        if old is None:
            FABRIC_CONTEXT.pop("data_contract_overrides", None)
        else:
            FABRIC_CONTEXT["data_contract_overrides"] = old

def contract_arguments():
    """Return configured arguments for authoritative contract persistence services."""
    return dict(config=CONFIG, env=ENV, spark_session=spark, context=FABRIC_CONTEXT,
                schema=metadata_table_physical_schema(CONFIG, "METADATA_DATA_CONTRACT"))


In [ ]:
def seed_contract(role, definitions, action="Block", processing=None):
    """Save/freeze a real draft, caching only successfully frozen exact versions."""
    key = json.dumps([role, definitions, action, processing], sort_keys=True)
    if key in CONTRACTS:
        return CONTRACTS[key]
    if role not in COLUMN_DOCS:
        raise RuntimeError(f"Reserve and register {role!r} before authoring a contract.")
    draft = create_contract_draft(table_id=OWNED[role]["table_id"], **contract_arguments())
    payload = json.loads(draft["contract_payload_json"])
    payload["table"]["columns"] = COLUMN_DOCS[role]
    payload["table"]["processing"] = processing or {"load_strategy": "overwrite"}
    column_ids = {doc["column_name"]: doc["column_id"] for doc in COLUMN_DOCS[role]}
    rules = []
    for index, definition in enumerate(definitions):
        rid = str(uuid.uuid5(uuid.NAMESPACE_URL, f"{RUN_NAMESPACE}/{key}/{index}"))
        row = canonical_guardrail_rule_record(dict(
            contract_id=draft["contract_id"], contract_version=draft["contract_version"],
            table_id=OWNED[role]["table_id"], guardrail_rule_id=rid, guardrail_version=1,
            rule_id=rid, guardrail_type=definition["kind"], rule_type=definition["rule_type"],
            column_id=column_ids.get(definition.get("column")), action=action, is_active=True,
            rule_parameters_json=json.dumps(definition["parameters"])), config=CONFIG, env=ENV)
        doc = {name: row[name] for name in ("guardrail_rule_id", "guardrail_version", "contract_id",
            "contract_version", "column_id", "guardrail_type", "rule_id", "rule_type", "action", "is_active")}
        doc["rule_parameters"] = json.loads(row["rule_parameters_json"])
        rules.append(doc)
    payload["guardrails"] = rules
    saved = save_contract_draft(draft=draft, payload=payload, **contract_arguments())
    frozen = freeze_contract(draft=saved, **contract_arguments())
    record = frozen["contract"]
    assert record["status"] == "frozen" and not record["is_active"]
    assert frozen["payload"]["guardrails"] == rules
    spec = dict(role=role, table_id=OWNED[role]["table_id"], contract_id=record["contract_id"],
                version=int(record["contract_version"]), rules=rules, record=record, payload=frozen["payload"])
    CONTRACTS[key] = spec
    return spec


## 03. Baselines and reporting

Every task is registered before its section executes. `run_case(test_id)` reruns exactly one task. `run_group(group)` runs a readable group. Failures do not prevent independent tasks or the final summary.

Preparation failures are BLOCKED with a traceback; assertion/runtime failures are FAIL. A check-level Block is a normal returned result with `can_continue=False`. Integration Block must raise at the expected stage and have the expected new persisted failed rule. A failed dependency never counts as a completed baseline.

For every check, the runner compares newly written result IDs with its pre-call snapshot, then requires exactly one row per expected rule/version/table/current activity and the expected status/continuation decision. It rejects extra or contradictory new outcomes. No old activity result can satisfy a new test.


In [ ]:
ACTIVITY_ID = build_runtime_audit_fields(config=CONFIG, env=ENV, runtime_context=FABRIC_CONTEXT)["_activity_id"]
STATUS = {"passed": "PASS", "pass": "PASS", "warning": "WARN", "warn": "WARN",
          "failed": "BLOCK", "fail": "BLOCK", "skipped": "SKIP", "baseline_created": "BASELINE"}
SUMMARY_COLUMNS = ["Test ID", "Guardrail", "Test scenario", "Read/Write boundary", "Contract version",
    "Expected status", "Actual status", "Expected rule evidence", "Actual rule evidence",
    "PASS/FAIL/SKIP/BLOCKED", "Diagnostic details"]

def normalized(value):
    """Translate supported runtime status labels into summary labels."""
    return STATUS.get(str(value).lower(), str(value).upper())

def require(condition, message):
    """Raise an assertion failure when an expected condition is false."""
    if not condition:
        raise AssertionError(message)

def register(test_id, group, guardrail, scenario, boundary, expected, prepare, call, verify,
             publication=False, dependencies=()):
    """Register a test task and initialize its verdict as unexecuted."""
    require(test_id not in TASKS, f"Duplicate Test ID: {test_id}")
    TASKS[test_id] = dict(group=group, prepare=prepare, call=call, verify=verify,
                         publication=publication, dependencies=dependencies)
    RESULTS[test_id] = dict(zip(SUMMARY_COLUMNS, [test_id, guardrail, scenario, boundary, "not prepared",
        expected, "NOT EXECUTED", "not prepared", "[]", "BLOCKED", "Registered; not executed."]))

def persisted_rows(specs):
    """Read current-activity results for the exact selected contract versions."""
    if not specs:
        return []
    selected = F.lit(False)
    for spec in specs:
        selected = selected | ((F.col("table_id") == spec["table_id"]) &
            (F.col("contract_id") == spec["contract_id"]) & (F.col("contract_version") == spec["version"]))
    return [r.asDict(recursive=True) for r in metadata("METADATA_GUARDRAIL_RESULTS").where(
        (F.col("environment_name") == ENV) & (F.col("_activity_id") == ACTIVITY_ID) & selected).collect()]

def expected_rows(spec, wanted, kinds=None):
    """Associate applicable frozen rules with their expected runtime status."""
    return [(spec, r, wanted) for r in spec["rules"] if kinds is None or r["guardrail_type"] in kinds]

def assert_evidence(rows, expectations):
    """Require exactly one correctly scoped and classified result per expected rule."""
    expected_keys = {(s["table_id"], s["contract_id"], s["version"], r["guardrail_rule_id"])
                     for s, r, wanted in expectations}
    actual_keys = [(r["table_id"], r["contract_id"], int(r["contract_version"]), r["guardrail_rule_id"]) for r in rows]
    require(Counter(actual_keys) == Counter(expected_keys), "Missing, extra, duplicate or wrong-version result rows")
    for spec, doc, wanted in expectations:
        row = next(r for r in rows if r["guardrail_rule_id"] == doc["guardrail_rule_id"])
        require(normalized(row["status"]) == wanted, f"Rule {doc['rule_id']}: expected {wanted}, got {row['status']}")
        require(row["can_continue"] is (wanted != "BLOCK"), "Persisted continuation decision is incorrect")
        require(row["guardrail_version"] == doc["guardrail_version"], "Wrong Guardrail version")
        require(row["execution_type"] == "enforce", "Wrong public-check execution type")


In [ ]:
def run_case(test_id, force=False):
    """Run preparation, call, verification and restoration as separately reported steps."""
    task, row = TASKS[test_id], RESULTS[test_id]
    if not force and ((TEST_FILTER and TEST_FILTER not in test_id) or
                      (task["publication"] and not INCLUDE_PUBLICATION_TESTS)):
        row.update({"PASS/FAIL/SKIP/BLOCKED": "SKIP", "Actual status": "SKIP",
                    "Diagnostic details": "Excluded by explicit notebook run configuration."})
        return
    missing = [dep for dep in task["dependencies"] if RESULTS[dep]["PASS/FAIL/SKIP/BLOCKED"] != "PASS"]
    if missing:
        row.update({"PASS/FAIL/SKIP/BLOCKED": "BLOCKED", "Actual status": "NOT EXECUTED",
                    "Diagnostic details": "Failed/unexecuted prerequisite: " + ", ".join(missing)})
        return
    artifact = dict(result=None, error=None, traceback="", new_results=[], stage_output="", contracts=[])
    ARTIFACTS[test_id] = artifact
    state = {}
    phase = "preparation"
    log = io.StringIO()
    preparation_log = io.StringIO()
    row.update({"PASS/FAIL/SKIP/BLOCKED": "BLOCKED", "Actual status": "NOT EXECUTED"})
    try:
        with redirect_stdout(preparation_log), redirect_stderr(preparation_log):
            state = task["prepare"]()
        specs = state.get("specs", [])
        artifact["state"] = state
        artifact["contracts"] = [s["payload"] for s in specs]
        row["Contract version"] = json.dumps([(s["contract_id"], s["version"]) for s in specs])
        row["Expected rule evidence"] = json.dumps([(doc["guardrail_rule_id"], wanted)
            for spec, doc, wanted in state.get("evidence", [])])
        before = {r["guardrail_result_id"] for r in persisted_rows(specs)}
        phase = "execution"
        try:
            with selection(specs, state.get("overrides")), redirect_stdout(log), redirect_stderr(log):
                artifact["result"] = task["call"](state)
        except Exception as exc:
            artifact.update(error=exc, traceback=traceback.format_exc())
        rows = [r for r in persisted_rows(specs) if r["guardrail_result_id"] not in before]
        artifact["new_results"] = rows
        row["Actual rule evidence"] = json.dumps(rows, default=str, sort_keys=True)
        row["Actual status"] = normalized((artifact["result"] or {}).get("status", "RETURNED")) if not artifact["error"] else "EXCEPTION"
        artifact["stage_output"] = log.getvalue()
        task["verify"](state, artifact)
        assert_evidence(rows, state.get("evidence", []))
        if artifact.get("actual_status"):
            row["Actual status"] = artifact["actual_status"]
        row["PASS/FAIL/SKIP/BLOCKED"] = "PASS"
        row["Diagnostic details"] = "Assertions passed."
    except Exception:
        row["PASS/FAIL/SKIP/BLOCKED"] = "BLOCKED" if phase == "preparation" else "FAIL"
        row["Diagnostic details"] = f"{phase}: " + traceback.format_exc() + artifact["traceback"]
    finally:
        artifact["stage_output"] = log.getvalue()
        artifact["preparation_output"] = preparation_log.getvalue()
        try:
            if state.get("restore"):
                state["restore"]()
            elif task["group"] in {"freshness", "drift", "reappearance", "read-integration", "write-integration", "coverage"}:
                put("source", clean_frame())
        except Exception:
            row["PASS/FAIL/SKIP/BLOCKED"] = "FAIL"
            row["Diagnostic details"] += "\nFixture restoration failed: " + traceback.format_exc()
    print(test_id, row["PASS/FAIL/SKIP/BLOCKED"], row["Actual status"])

def run_group(group):
    """Execute registered tasks belonging to one named notebook section."""
    for test_id, task in TASKS.items():
        if task["group"] == group:
            run_case(test_id)

def returned(artifact):
    """Require a successful dictionary return and expose it for assertions."""
    require(artifact["error"] is None, "Unexpected runtime exception: " + artifact["traceback"])
    require(isinstance(artifact["result"], dict), "Public function did not return a result dictionary")
    return artifact["result"]

def assert_status(result, wanted):
    """Check the returned status and its continuation decision together."""
    require(normalized(result["status"]) == wanted, f"Expected {wanted}, got {result['status']}")
    require(result["can_continue"] is (wanted != "BLOCK"), "Returned continuation decision is incorrect")


In [ ]:
def check_call(kind, frame, spec, **kwargs):
    """Call the selected public check with the owned table identity."""
    common = dict(table_id=spec["table_id"], spark_session=spark, verbose=False)
    functions = {"data_quality": check_dq, "schema": check_schema, "sensitive_data": check_sensitive_data}
    if kind == "data_quality":
        kwargs["row_identity_columns"] = ["row_id"]
    return functions[kind](frame, **common, **kwargs)

def prepare_check(definitions, action="Block", frame_options=None, expected="PASS", role="checks", **extra):
    """Freeze the selected definitions and construct an independent input fixture."""
    spec = seed_contract(role, definitions, action)
    frame = make_frame(**(frame_options or {}))
    return dict(specs=[spec], spec=spec, frame=frame, wanted=expected,
                evidence=expected_rows(spec, expected), **extra)

def verify_basic(state, artifact):
    """Verify the expected status of a simple public-check result."""
    assert_status(returned(artifact), state["wanted"])

# The check baselines do not depend on a successful orchestrator or publication.
for kind, definition in (("schema", SCHEMA), ("data_quality", rule("data_quality", "completeness", {
        "columns": ["name"], "maximum_missing_percent": 0, "treat_blank_as_missing": True})),
        ("sensitive_data", rule("sensitive_data", "remove", {"scope": "column", "treatment": "remove"}, "secret"))):
    register(f"baseline/{kind}", "baseline", kind, "Clean contract-backed public check", "Check", "PASS",
        lambda d=definition: prepare_check([d]),
        lambda s, k=kind: check_call(k, s["frame"], s["spec"]), verify_basic)
run_group("baseline")


## 04. Individual Guardrail checks

The matrix is explicit and finite. Each configured rule runs with **Warn and Block**, clean input, each listed input variant, and a restored clean replay. DQ and Sensitive Data checks return their results even for Block; separate integration tests assert exceptions.

`row_id` remains stable while business keys change. Every DQ variant declares expected failed row IDs, checked count, percent and involved columns. Every Sensitive Data variant asserts the row multiset, unchanged columns, exact prepared values and mapping constraints. Combined cases also verify that a failed treatment preserves successful independent treatments. No checks call evaluator cores or use mocks.


### 04A. Data Quality — declare the behavior matrix

Completeness covers blank handling and 0/25/100% thresholds; uniqueness covers single/composite keys and 50/75/100% thresholds. Value-set rules cover allow/block, nulls and all-invalid input. Range covers lower/upper/both bounds and all inclusivity pairs. Relationships cover all six operators with equal/less/greater/one-null/both-null inputs. Conditional rules cover both operators, matching/nonmatching/null conditions, blank handling and allow/block modes. Custom Expression covers boolean comparison, methods, compound predicates, nulls and invalid/nonboolean expressions. All valid definitions also run on empty data.


In [ ]:
DQ_MATRIX = []

def dq_case(name, rule_type, parameters, variants, clean_options=None):
    """Each variant is (label, frame options, exact failed row IDs)."""
    DQ_MATRIX.append(dict(name=name, definition=rule("data_quality", rule_type, parameters),
        variants=[("clean", clean_options or {}, [])] + variants + [("empty", {"keep_ids": []}, []),
                                                                 ("restored", clean_options or {}, [])]))

for blank in (False, True):
    for threshold in (0, 25, 100):
        params = dict(columns=["name"], maximum_missing_percent=threshold, treat_blank_as_missing=blank)
        dq_case(f"completeness/blank={blank}/max={threshold}", "completeness", params, [
            ("one-null", {"changes": {1: {"name": None}}}, [1] if threshold < 25 else []),
            ("one-blank", {"changes": {1: {"name": "  "}}}, [1] if blank and threshold < 25 else []),
            ("all-null", {"changes": {i: {"name": None} for i in range(1, 5)}}, list(range(1, 5)) if threshold < 100 else []),
        ])
for columns in (["business_key"], ["category", "code"]):
    for threshold in (50, 75, 100):
        duplicate = {2: {"business_key": "K1"}} if len(columns) == 1 else {2: {"category": "A", "code": "AA01"}}
        dq_case(f"uniqueness/{'+'.join(columns)}/min={threshold}", "uniqueness",
            dict(columns=columns, minimum_unique_percent=threshold), [
                ("duplicate-pair", {"changes": duplicate}, [1, 2] if threshold > 75 else []),
                ("all-same", {"changes": {i: {c: "X" for c in columns} for i in range(1, 5)}}, [1, 2, 3, 4]),
                ("null-pair", {"changes": {i: {c: None for c in columns} for i in (1, 2)}}, [1, 2] if threshold > 75 else []),
            ])
for mode in ("allow", "block"):
    values = ["A", "B"] if mode == "allow" else ["DENY"]
    bad = "X" if mode == "allow" else "DENY"
    dq_case(f"value_set/{mode}", "value_set", dict(columns=["category"], mode=mode, values=values), [
        ("one-invalid", {"changes": {1: {"category": bad}}}, [1]),
        ("null", {"changes": {1: {"category": None}}}, []),
        ("all-invalid", {"changes": {i: {"category": bad} for i in range(1, 5)}}, [1, 2, 3, 4]),
    ])


In [ ]:
for bounds in ("lower", "upper", "both"):
    for low_inclusive in (False, True):
        for high_inclusive in (False, True):
            if bounds == "lower" and not high_inclusive or bounds == "upper" and not low_inclusive:
                continue  # Do not duplicate an option that does not apply to this bound.
            params = dict(columns=["amount"], minimum_inclusive=low_inclusive, maximum_inclusive=high_inclusive)
            if bounds != "upper":
                params["minimum"] = 0
            if bounds != "lower":
                params["maximum"] = 100
            variants = [("null", {"changes": {1: {"amount": None}}}, [])]
            for label, value, fails in (("below", -1.0, bounds != "upper"),
                    ("lower-bound", 0.0, bounds != "upper" and not low_inclusive),
                    ("upper-bound", 100.0, bounds != "lower" and not high_inclusive),
                    ("above", 101.0, bounds != "lower")):
                variants.append((label, {"changes": {1: {"amount": value}}}, [1] if fails else []))
            dq_case(f"range/{bounds}/{low_inclusive}/{high_inclusive}", "range", params, variants)

dq_case("pattern", "pattern", dict(columns=["code"], pattern="^[A-Z]{2}[0-9]{2}$"), [
    ("invalid", {"changes": {1: {"code": "invalid"}}}, [1]),
    ("blank", {"changes": {1: {"code": ""}}}, [1]),
    ("null", {"changes": {1: {"code": None}}}, []),
])
# This truth table is explicit expected behavior, independent of Spark predicate construction.
RELATION_FAILURES = {
    "=":  {"less", "greater", "left-null", "right-null"},
    "!=": {"equal", "both-null"},
    "<":  {"equal", "greater", "left-null", "right-null"},
    "<=": {"greater", "left-null", "right-null"},
    ">":  {"equal", "less", "left-null", "right-null"},
    ">=": {"less", "left-null", "right-null"},
}
RELATION_INPUTS = {"less": (20.0, 100.0), "equal": (100.0, 100.0), "greater": (120.0, 100.0),
                   "left-null": (None, 100.0), "right-null": (100.0, None), "both-null": (None, None)}
for operator, failing_labels in RELATION_FAILURES.items():
    good = 100.0 if operator == "=" else (120.0 if operator in (">", ">=") else 20.0)
    clean_options = {"changes": {i: {"amount": good} for i in range(1, 5)}}
    variants = []
    for label, (left, right) in RELATION_INPUTS.items():
        changes = {i: {"amount": good} for i in range(1, 5)}
        changes[1] = {"amount": left, "ceiling": right}
        variants.append((label, {"changes": changes}, [1] if label in failing_labels else []))
    dq_case("relationship/" + operator, "column_relationship",
            dict(columns=["amount", "ceiling"], operator=operator), variants, clean_options)


In [ ]:
for operator in ("=", "!="):
    condition_value = "A" if operator == "=" else "B"
    for blank in (False, True):
        dq_case(f"conditional-completeness/{operator}/blank={blank}", "conditional_completeness",
            dict(columns=["category", "required_note"], condition_operator=operator,
                 condition_value=condition_value, treat_blank_as_missing=blank), [
                ("matching-null", {"changes": {1: {"required_note": None}}}, [1]),
                ("matching-blank", {"changes": {1: {"required_note": " "}}}, [1] if blank else []),
                ("nonmatching-null", {"changes": {2: {"required_note": None}}}, []),
                ("null-condition", {"changes": {1: {"category": None, "required_note": None}}}, [1] if operator == "!=" else []),
            ])
    for mode in ("allow", "block"):
        params = dict(columns=["category", "code"], condition_operator=operator,
                      condition_value=condition_value, mode=mode, values=["AA01", "CC03"] if mode == "allow" else ["ZZ99"])
        dq_case(f"conditional-values/{operator}/{mode}", "conditional_values", params, [
            ("matching-invalid", {"changes": {1: {"code": "ZZ99"}}}, [1]),
            ("nonmatching-invalid", {"changes": {2: {"code": "ZZ99"}}}, []),
            ("null-target", {"changes": {1: {"code": None}}}, []),
            ("null-condition", {"changes": {1: {"category": None, "code": "ZZ99"}}}, [1] if operator == "!=" else []),
        ])
for name, expression, variants in (
    ("comparison", "F.col('amount') >= F.lit(0)", [("negative", {"changes": {1: {"amount": -1.0}}}, [1]),
                                                ("null", {"changes": {1: {"amount": None}}}, [])]),
    ("method", "F.col('category').isin('A', 'B')", [("invalid", {"changes": {1: {"category": "X"}}}, [1])]),
    ("compound", "(F.col('category') != F.lit('A')) | F.col('required_note').isNotNull()",
        [("matching-null", {"changes": {1: {"required_note": None}}}, [1]),
         ("nonmatching-null", {"changes": {2: {"required_note": None}}}, [])]),
):
    dq_case("custom/" + name, "custom_expression", dict(expression_language="pyspark", expression=expression), variants)


### 04A. Data Quality — exact assertions and execution

A completeness/uniqueness tolerance applies to the whole dataset; tolerated invalid rows are therefore expected to produce no failures. For composite uniqueness all duplicated records must be reported. Null behavior is recorded explicitly rather than treated as a universal failure rule.

Failure-detail assertions compare a **multiset**, including rule, row identity, column name/role, raw value and type. They verify both returned and persisted counts/percentages. A checker identifying the wrong rows cannot pass.


In [ ]:
def verify_dq(state, artifact):
    """Verify exact DQ failures, percentages, detail rows and record-level tags."""
    result = returned(artifact)
    assert_status(result, state["wanted"])
    spec = state["spec"]
    expected = state["failed_ids_by_rule"]
    checks = {c["rule_id"]: c for c in result["checks"]}
    require(len(checks) == len(spec["rules"]) == len(result["checks"]), "Missing or duplicate DQ checks")
    records = {r["row_id"]: r for r in state["frame"].collect()}
    total = len(records)
    failed_union = set()
    detail_expected = Counter()
    dtypes = dict(state["frame"].dtypes)
    for doc in spec["rules"]:
        rid = doc["rule_id"]
        ids = expected[rid]
        failed_union.update(ids)
        c = checks[rid]
        wanted = ("WARN" if doc["action"] == "Warn" else "BLOCK") if ids else "PASS"
        require(normalized(c["status"]) == wanted and c["passed"] is (not ids), "Wrong per-rule outcome")
        require(c["failed_count"] == len(ids) and c["total_count"] == total, "Wrong per-rule counts")
        percent = round(100 * len(ids) / total, 4) if total else 0.0
        require(math.isclose(c["failed_percent"], percent, abs_tol=0.0001), "Wrong failure percentage")
        persisted = next(r for r in artifact["new_results"] if r["rule_id"] == rid)
        payload = json.loads(persisted["result_payload_json"])
        require(payload["failed_count"] == len(ids) and payload["total_count"] == total, "Persisted counts differ")
        require(math.isclose(payload["failed_percent"], percent, abs_tol=0.0001), "Persisted percent differs")
        columns = doc["rule_parameters"].get("columns", c["columns"])
        for index, column in enumerate(columns):
            role = ("left" if index == 0 else "right") if doc["rule_type"] == "column_relationship" else (
                   "participating" if doc["rule_type"] == "uniqueness" and len(columns) > 1 else "target")
            for row_id in ids:
                value = records[row_id][column]
                detail_expected[(rid, row_id, column, role, None if value is None else str(value), dtypes[column])] += 1
    actual = Counter((r["rule_id"], json.loads(r["row_identity"])["row_id"], r["column_name"],
                      r["column_role"], r["raw_value"], r["raw_value_type"]) for r in result["failed_values"].collect())
    require(actual == detail_expected, "Wrong failed-value rows, columns, roles, raw values or duplicates")
    summary = result["summary"]
    require(summary["DQ_RULE_COUNT"] == len(spec["rules"]), "Wrong summary rule count")
    require(summary["DQ_FAILED_ROW_COUNT"] == len(failed_union), "Wrong distinct failed row count")
    require(math.isclose(summary["DQ_FAILED_ROW_PERCENT"], round(100 * len(failed_union) / total, 4) if total else 0.0, abs_tol=0.0001), "Wrong summary percentage")
    tagged = result["dataframe"].collect()
    require(Counter(r["row_id"] for r in tagged) == Counter(records.keys()), "DQ tagging changed row multiplicity")
    for row in tagged:
        failed_rules = sorted(rid for rid, ids in expected.items() if row["row_id"] in ids)
        require(row["_dq_failed_rules"] == ",".join(failed_rules), "Wrong per-row failed-rule tags")
        wanted = state["wanted"] if failed_rules else "PASS"
        require(normalized(row["_dq_check_status"]) == wanted, "Wrong per-row status tag")


In [ ]:
def prepare_dq(definition, action, options, failed_ids):
    """Attach explicit expected failed record IDs to a DQ fixture."""
    wanted = ("WARN" if action == "Warn" else "BLOCK") if failed_ids else "PASS"
    state = prepare_check([definition], action, options, wanted)
    rid = state["spec"]["rules"][0]["rule_id"]
    state["failed_ids_by_rule"] = {rid: list(failed_ids)}
    return state

for item in DQ_MATRIX:
    for action in ("Warn", "Block"):
        for label, options, failed_ids in item["variants"]:
            test_id = f"dq/{item['name']}/{action}/{label}"
            expected = ("WARN" if action == "Warn" else "BLOCK") if failed_ids else "PASS"
            register(test_id, "dq", item["name"], label, "Check", expected,
                lambda d=item["definition"], a=action, o=options, f=failed_ids: prepare_dq(d, a, o, f),
                lambda s: check_call("data_quality", s["frame"], s["spec"]), verify_dq)
print("DQ definitions:", len(DQ_MATRIX), "registered DQ cases:", sum(t["group"] == "dq" for t in TASKS.values()))
run_group("dq")


### 04B. Schema

All four presets × both actions × clean/reordered/empty/missing/extra/type/mixed/restored input. Assertions identify the exact missing/unexpected columns and type mismatch column. Schema monitoring continues with warning; a Warn action must never block.


In [ ]:
SCHEMA_VARIANTS = {
    "clean": ([], [], []), "reordered": ([], [], []), "empty": ([], [], []),
    "missing": (["secret"], [], []), "extra": ([], ["unexpected"], []),
    "type": ([], [], ["amount"]), "mixed": (["secret"], ["unexpected"], ["amount"]),
    "restored": ([], [], []),
}

def prepare_schema(mode, action, variant):
    """Construct a schema variant and declare its expected differences."""
    missing, extra, mismatch = SCHEMA_VARIANTS[variant]
    differences = bool(missing or extra or mismatch)
    blocking = bool(missing or mismatch or mode == "strict" and extra)
    wanted = "PASS" if not differences else ("BLOCK" if blocking and mode != "skip" and action == "Block" else "WARN")
    state = prepare_check([rule("schema", mode)], action, expected=wanted)
    data = state["frame"]
    if missing:
        data = data.drop(*missing)
    if extra:
        data = data.withColumn("unexpected", F.lit(1))
    if mismatch:
        data = data.withColumn("amount", F.col("amount").cast("string"))
    if variant == "reordered":
        data = data.select(*reversed(data.columns))
    if variant == "empty":
        data = data.limit(0)
    state.update(frame=data, missing=missing, extra=extra, mismatch=mismatch)
    return state

def verify_schema(state, artifact):
    """Compare exact missing, unexpected and mismatched schema columns."""
    result = returned(artifact)
    assert_status(result, state["wanted"])
    require(sorted(result["missing_columns"]) == state["missing"], "Wrong missing columns")
    require(sorted(result["unexpected_columns"]) == state["extra"], "Wrong unexpected columns")
    require(sorted(m["column"] for m in result["datatype_mismatches"]) == state["mismatch"], "Wrong mismatched types")

for mode in ("strict", "minimum_required", "relaxed", "skip"):
    for action in ("Warn", "Block"):
        for variant, (missing, extra, mismatch) in SCHEMA_VARIANTS.items():
            blocking = bool(missing or mismatch or mode == "strict" and extra)
            expected = "PASS" if not (missing or extra or mismatch) else (
                "BLOCK" if blocking and mode != "skip" and action == "Block" else "WARN")
            register(f"schema/{mode}/{action}/{variant}", "schema", "Schema", variant, "Check", expected,
                lambda m=mode, a=action, v=variant: prepare_schema(m, a, v),
                lambda s: check_call("schema", s["frame"], s["spec"]), verify_schema)
run_group("schema")


### 04C. Sensitive Data

Mask covers independent leading/trailing preservation, short/empty/null strings and a multicharacter mask. Tokenize covers repeated/null originals, mapping reuse and invalid/conflicting mappings. Bucket covers open ends, exact lower boundaries, nulls and nonnumeric input. Remove covers nulls and missing input. All treatments cover empty data, missing columns, both actions, restoration, and a combined four-treatment case.

Successful treatments compare complete row multisets; unique-ID checks occur **before** dictionary lookup. No row duplication can be hidden. Token mappings remain in memory and are never written to a business or Metadata table.


In [ ]:
MASK_OPTIONS = [
    (1, 0, "*", ["S********", "S1", None, ""]),
    (0, 3, "*", ["******67A", "S1", None, ""]),
    (1, 3, "*", ["S*****67A", "S1", None, ""]),
    (0, 0, "*", ["*********", "**", None, ""]),
    (1, 1, "*", ["S*******A", "S1", None, ""]),
    (1, 1, "XY", ["SXYXYXYXYXYXYXYA", "S1", None, ""]),
]
SENSITIVE_DEFS = {
    "mask": rule("sensitive_data", "mask", dict(scope="column", treatment="mask", preserve_start=1, preserve_end=1, mask_character="*"), "email"),
    "tokenize": rule("sensitive_data", "tokenize", dict(scope="column", treatment="tokenize"), "email"),
    "bucket": rule("sensitive_data", "bucket", dict(scope="column", treatment="bucket", bins=[0, 30, 60], labels=["young", "middle", "older"]), "age"),
    "remove": rule("sensitive_data", "remove", dict(scope="column", treatment="remove"), "secret"),
}
SENSITIVE_MATRIX = []
for start, end, character, expected in MASK_OPTIONS:
    definition = rule("sensitive_data", "mask", dict(scope="column", treatment="mask",
        preserve_start=start, preserve_end=end, mask_character=character), "email")
    SENSITIVE_MATRIX.append(dict(name=f"mask/{start}/{end}/{character}", definitions=[definition],
        changes={i: {"email": value} for i, value in enumerate(["S1234567A", "S1", None, ""], 1)},
        expected_updates={i: {"email": value} for i, value in enumerate(expected, 1)}))
for treatment, definition in SENSITIVE_DEFS.items():
    variants = ["clean", "empty", "missing", "restored"]
    if treatment == "remove":
        variants += ["null"]
    if treatment == "tokenize":
        variants += ["null", "reuse", "conflicting-original", "conflicting-token", "null-mapping", "invalid-mapping-schema"]
    if treatment == "bucket":
        variants += ["lower-end", "upper-end", "nonnumeric"]
    for variant in variants:
        SENSITIVE_MATRIX.append(dict(name=f"{treatment}/{variant}", definitions=[definition], variant=variant))
SENSITIVE_MATRIX.append(dict(name="combined", definitions=[
    rule("sensitive_data", "mask", dict(scope="column", treatment="mask", preserve_start=1, preserve_end=1, mask_character="*"), "name"),
    SENSITIVE_DEFS["tokenize"], SENSITIVE_DEFS["bucket"], SENSITIVE_DEFS["remove"]], variant="combined"))
SENSITIVE_MATRIX.append({**SENSITIVE_MATRIX[-1], "name": "combined/missing-mask-column", "variant": "partial"})


In [ ]:
def prepare_sensitive(item, action):
    """Build treatment inputs and optional caller-owned token mapping fixtures."""
    variant = item.get("variant", "mask-options")
    failures = {"missing", "nonnumeric", "conflicting-original", "conflicting-token", "null-mapping", "invalid-mapping-schema", "partial"}
    wanted = ("WARN" if action == "Warn" else "BLOCK") if variant in failures else "PASS"
    changes = item.get("changes", {})
    if variant == "null":
        changes = {2: {item["definitions"][0]["column"]: None}}
    if variant == "lower-end":
        changes = {i: {"age": v} for i, v in enumerate([-1, 0, 29, 30], 1)}
    if variant == "upper-end":
        changes = {i: {"age": v} for i, v in enumerate([59, 60, 600, None], 1)}
    state = prepare_check(item["definitions"], action, {"changes": changes, "keep_ids": [] if variant == "empty" else None}, wanted)
    data = state["frame"]
    if variant == "partial":
        data = data.drop("name")
    if variant == "missing":
        data = data.drop(item["definitions"][0]["column"])
    if variant == "nonnumeric":
        data = data.withColumn("age", F.col("age").cast("string"))
    mapping = None
    if variant in {"reuse", "conflicting-original", "conflicting-token", "null-mapping", "invalid-mapping-schema"}:
        cid = next(d["column_id"] for d in COLUMN_DOCS["checks"] if d["column_name"] == "email")
        prefix = [state["spec"]["table_id"], cid]
        mapping_rows = [prefix + ["alpha@example.test", "established-token"]]
        if variant == "conflicting-original":
            mapping_rows += [prefix + ["alpha@example.test", "different-token"]]
        if variant == "conflicting-token":
            mapping_rows += [prefix + ["beta@example.test", "established-token"]]
        if variant == "null-mapping":
            mapping_rows = [prefix + ["alpha@example.test", None]]
        mapping = spark.createDataFrame(mapping_rows, "table_id string, column_id string, original_value string, token_value string")
        if variant == "invalid-mapping-schema":
            mapping = mapping.drop("column_id")
    if variant == "partial":
        state["evidence"] = [(state["spec"], doc, wanted if doc["rule_type"] == "mask" else "PASS") for doc in state["spec"]["rules"]]
    state.update(frame=data, mapping=mapping, item=item, variant=variant)
    return state

def row_multiset(rows):
    """Represent complete rows as a multiset without hiding duplicates."""
    return Counter(json.dumps(r.asDict() if hasattr(r, "asDict") else r, sort_keys=True, default=str) for r in rows)


In [ ]:
def verify_sensitive(state, artifact):
    """Verify treatment outcomes, row preservation, prepared values and mappings."""
    result = returned(artifact)
    assert_status(result, state["wanted"])
    before_rows, after_rows = state["frame"].collect(), result["dataframe"].collect()
    require(len(before_rows) == len(after_rows), "Sensitive treatment changed row count")
    before_ids, after_ids = Counter(r["row_id"] for r in before_rows), Counter(r["row_id"] for r in after_rows)
    require(before_ids == after_ids and all(n == 1 for n in before_ids.values()), "Duplicated or missing row IDs")
    require(len(result["checks"]) == len(state["spec"]["rules"]), "Wrong treatment check count")
    expected_checks = Counter((doc["guardrail_rule_id"], wanted) for spec, doc, wanted in state["evidence"])
    actual_checks = Counter((c["guardrail_rule_id"], normalized(c["status"])) for c in result["checks"])
    require(actual_checks == expected_checks, "Wrong returned per-treatment outcomes")
    if state["wanted"] != "PASS" and state["variant"] != "partial":
        require(row_multiset(before_rows) == row_multiset(after_rows), "Failed treatment changed input")
        require(result["support_mapping"] is None, "Failed treatment returned a mapping")
        return
    expected = {r["row_id"]: r.asDict() for r in before_rows}
    mapping = result["support_mapping"]
    lookup = {}
    if any(d["rule_type"] == "tokenize" for d in state["item"]["definitions"]):
        require(mapping is not None, "Token mapping missing")
        rows = mapping.collect()
        originals = {r["email"] for r in before_rows if r["email"] is not None}
        require(len(rows) == len(originals), "Wrong mapping row count or duplicate originals")
        lookup = {r["original_value"]: r["token_value"] for r in rows}
        require(set(lookup) == originals and len(set(lookup.values())) == len(lookup), "Invalid token mapping cardinality")
        cid = next(d["column_id"] for d in COLUMN_DOCS["checks"] if d["column_name"] == "email")
        for row in rows:
            require(row["table_id"] == state["spec"]["table_id"] and row["column_id"] == cid, "Wrong mapping scope")
            require(row["token_value"] != row["original_value"], "Token retained original")
            if state["variant"] == "reuse" and row["original_value"] == "alpha@example.test":
                require(row["token_value"] == "established-token", "Existing token was not reused")
            else:
                uuid.UUID(row["token_value"])
    else:
        require(mapping is None, "Non-token treatment returned a mapping")
    for row_id, row in expected.items():
        for definition in state["item"]["definitions"]:
            treatment, column = definition["rule_type"], definition["column"]
            if state["variant"] == "partial" and treatment == "mask":
                continue
            if treatment == "remove":
                del row[column]
            elif treatment == "tokenize":
                row[column] = None if row[column] is None else lookup[row[column]]
            elif treatment == "mask":
                if state["variant"] == "combined":
                    row[column] = {1: "A***e", 2: "Bob", 3: "C**a", 4: "Dan"}[row_id]
                elif "expected_updates" in state["item"]:
                    row.update(state["item"]["expected_updates"][row_id])
                else:
                    row[column] = {1: "a****************t", 2: "b***************t", 3: "a****************t", 4: "g****************t"}[row_id]
            elif treatment == "bucket":
                values = {"lower-end": ["young", "young", "young", "middle"],
                          "upper-end": ["middle", "older", "older", None]}.get(state["variant"], ["young", "middle", "older", "older"])
                row[column] = values[row_id - 1]
    removed = {d["column"] for d in state["item"]["definitions"] if d["rule_type"] == "remove"}
    require(set(result["dataframe"].columns) == set(state["frame"].columns) - removed, "Wrong prepared columns")
    require(row_multiset(expected.values()) == row_multiset(after_rows), "Wrong values or changed unrelated columns")
    persisted = json.dumps(artifact["new_results"], default=str)
    for row in before_rows:
        for definition in state["item"]["definitions"]:
            value = row.asDict().get(definition["column"])
            if isinstance(value, str) and len(value) > 6:
                require(value not in persisted, "Raw sensitive value leaked into Metadata result")


In [ ]:
for item in SENSITIVE_MATRIX:
    for action in ("Warn", "Block"):
        failing = item.get("variant") in {"missing", "nonnumeric", "conflicting-original", "conflicting-token", "null-mapping", "invalid-mapping-schema", "partial"}
        expected = ("WARN" if action == "Warn" else "BLOCK") if failing else "PASS"
        register(f"sensitive/{item['name']}/{action}", "sensitive", "Sensitive Data", item["name"], "Check", expected,
            lambda i=item, a=action: prepare_sensitive(i, a),
            lambda s: check_call("sensitive_data", s["frame"], s["spec"], existing_mapping=s["mapping"]), verify_sensitive)
run_group("sensitive")


### 04D. Freshness

Real `pipeline_read()` calls capture the source observation before `check_freshness()`. Minutes/hours/days are checked on each side of the cutoff, plus lag-days, null, empty, static, clean and restored cases, with both actions. Cutoffs have a buffer for Spark work; the assertion also checks the returned required-minimum timestamp against the call's actual start/end times. No clock is patched.


In [ ]:
FRESHNESS_MATRIX = [
    ("minutes", {"maximum_age": 10, "maximum_age_unit": "minutes"}, 600),
    ("hours", {"maximum_age": 1, "maximum_age_unit": "hours"}, 3600),
    ("days", {"maximum_age": 1, "maximum_age_unit": "days"}, 86400),
    ("lag-days", {"max_lag_days": 2}, 172800),
]

def read_physical(role="source", orchestrated=False):
    """Read an owned source through the public reader or Read orchestrator."""
    s = OWNED[role]
    function = orchestrate_read if orchestrated else pipeline_read
    kwargs = dict(store=s["store"], schema=s["schema"], table_name=s["name"], spark_session=spark, verbose=True)
    if orchestrated:
        kwargs["frequency_enabled"] = False
    return function(**kwargs)

def prepare_freshness(parameters, seconds, action, variant):
    """Freeze freshness intent and retain restoration and cutoff expectations."""
    static = variant == "static"
    definition = rule("freshness", "skip" if static else "max_age", {"refresh_expectation": "static"} if static else
                      {"freshness_column": "modified_at", **parameters})
    wanted = "SKIP" if static else (("WARN" if action == "Warn" else "BLOCK") if variant in {"stale", "null", "empty"} else "PASS")
    spec = seed_contract("source", [definition], action)
    return dict(specs=[spec], spec=spec, wanted=wanted, seconds=seconds, variant=variant,
                evidence=[] if static else expected_rows(spec, wanted), restore=lambda: put("source", clean_frame()))

def call_freshness(state):
    """Capture a real source observation and execute the public Freshness check."""
    state["start"] = datetime.now()
    margin = min(300, state["seconds"] / 2)
    age = state["seconds"] + margin if state["variant"] == "stale" else state["seconds"] - margin
    value = None if state["variant"] == "null" else state["start"] - timedelta(seconds=age)
    changes = {i: {"modified_at": value} for i in range(1, 5)}
    if state["variant"] == "mixed":
        changes.update({i: {"modified_at": state["start"] - timedelta(seconds=state["seconds"] + margin)} for i in (2, 3, 4)})
    state["expected_latest"] = value
    put("source", make_frame(changes, keep_ids=[] if state["variant"] == "empty" else None))
    read_physical()
    result = check_freshness(state["spec"]["table_id"], spark_session=spark, verbose=False)
    state["end"] = datetime.now()
    return result

def verify_freshness(state, artifact):
    """Verify freshness status, maximum value and the elapsed-time cutoff."""
    result = returned(artifact)
    assert_status(result, state["wanted"])
    if state["variant"] in {"static", "null", "empty"}:
        return
    latest = datetime.fromisoformat(result["latest_value"])
    minimum = datetime.fromisoformat(result["required_min_value"])
    low, high = state["start"] - timedelta(seconds=state["seconds"]), state["end"] - timedelta(seconds=state["seconds"])
    if "max_lag_days" in state["spec"]["rules"][0]["rule_parameters"]:
        require(low.date() <= minimum.date() <= high.date(), "Lag-days cutoff is incorrect")
        require(latest.date() == state["expected_latest"].date(), "Wrong maximum freshness date")
    else:
        require(low <= minimum <= high, "Freshness unit conversion/cutoff is incorrect")
        require(latest == state["expected_latest"], "Wrong maximum freshness timestamp")

for unit, parameters, seconds in FRESHNESS_MATRIX:
    for action in ("Warn", "Block"):
        for variant in ("clean", "stale", "mixed", "null", "empty", "restored"):
            expected = ("WARN" if action == "Warn" else "BLOCK") if variant in {"stale", "null", "empty"} else "PASS"
            register(f"freshness/{unit}/{action}/{variant}", "freshness", "Freshness", variant, "Check", expected,
                lambda p=parameters, sec=seconds, a=action, v=variant: prepare_freshness(p, sec, a, v), call_freshness, verify_freshness)
for action in ("Warn", "Block"):
    register(f"freshness/static/{action}", "freshness", "Freshness", "Static explicitly skips age evaluation", "Check", "SKIP",
        lambda a=action: prepare_freshness({}, 600, a, "static"), call_freshness, verify_freshness)
run_group("freshness")


### 04E. Source Drift

Source Drift needs a real accepted source-target baseline. Baseline tasks use `pipeline_read()` → `check_source_drift()` → `pipeline_write()`; no Source Observation history is injected. Each processing strategy/action owns a separate target. Check-only mutations do not advance the baseline or target Delta version.

Append must Warn/Block for historical changes, deletions and reappearance; new partitions are compatible. Overwrite/SCD1/SCD2 can reconcile these changes. Assertions identify exact changed/new/removed/reappeared partitions. Publication-dependent cases are explicitly SKIP when `INCLUDE_PUBLICATION_TESTS=False`.


In [ ]:
def target_version(role):
    """Return the latest Delta version of one owned target."""
    return int(DeltaTable.forPath(spark, OWNED[role]["path"]).history(1).select("version").first()[0])

def publication_state(source_spec, target_spec):
    """Snapshot persisted Source Observation rows for an exact owned relationship."""
    ids = (source_spec["table_id"], target_spec["table_id"])
    rows = metadata("METADATA_SOURCE_OBSERVATION").where((F.col("environment_name") == ENV) &
        (F.col("source_table_id") == ids[0]) & (F.col("target_table_id") == ids[1])).collect()
    return row_multiset(rows)

def write_baseline(frame, spec, source_ids):
    """Publish through the public writer using the frozen processing definition."""
    s = OWNED[spec["role"]]
    kwargs = dict(store=s["store"], schema=s["schema"], table_name=s["name"], spark_session=spark)
    processing = spec["payload"]["table"]["processing"]
    parameters = {k: v for k, v in processing.items() if k != "load_strategy"}
    return pipeline_write(frame, load_strategy=processing["load_strategy"],
        load_strategy_parameters=parameters, source_table_ids=source_ids, **kwargs)

def publish(frame, spec, source_ids, mode="validate"):
    """Execute the real Write orchestrator with explicit sources and contract mode."""
    s = OWNED[spec["role"]]
    kwargs = dict(store=s["store"], schema=s["schema"], table_name=s["name"], spark_session=spark)
    return orchestrate_write(frame, sources=[{"table_id": tid} for tid in source_ids], write_mode="overwrite",
        contracts={"tables": {spec["table_id"]: {"mode": mode}}}, frequency_enabled=False, verbose=True, **kwargs)

DRIFT_PROCESSING = {
    "append": {"load_strategy": "append"}, "overwrite": {"load_strategy": "overwrite"},
    "scd1": {"load_strategy": "scd1", "key_columns": ["business_key"]},
    "scd2": {"load_strategy": "scd2", "key_columns": ["business_key"], "effective_column": "modified_at"},
}

def prepare_drift_baseline(strategy, action, target_strategy="overwrite"):
    # New target on a deliberate baseline rerun, so first-observation assertions remain valid.
    """Reserve a fresh target and freeze the source and target processing contracts."""
    role = "drift_" + uuid.uuid4().hex[:10]
    reserve(role, "Silver", publication=True)
    source = seed_contract("source", [SCHEMA, FRESH, DRIFT], action, DRIFT_PROCESSING[strategy])
    target_processing = {"load_strategy": target_strategy}
    if target_strategy == "scd1":
        target_processing["key_columns"] = ["business_key"]
    target = seed_contract(role, [SCHEMA], processing=target_processing)
    return dict(specs=[source, target], source=source, target=target, role=role,
        wanted="BASELINE", evidence=expected_rows(source, "BASELINE", {"source_drift"}),
        previous_version=target_version(role), restore=lambda: put("source", clean_frame()))

def call_drift_baseline(state):
    """Observe the source and establish an accepted baseline through a real write."""
    put("source", clean_frame())
    read_physical()
    result = check_source_drift(state["source"]["table_id"], target_table_id=state["target"]["table_id"],
                               spark_session=spark, verbose=False)
    state["publication"] = write_baseline(clean_frame(), state["target"], [state["source"]["table_id"]])
    return result

def verify_drift_baseline(state, artifact):
    """Verify first-observation status, target publication and accepted history."""
    assert_status(returned(artifact), "BASELINE")
    require(target_version(state["role"]) > state["previous_version"], "Baseline did not publish")
    require(any(json.loads(row)["observation_status"] == "drift_committed" for row in publication_state(state["source"], state["target"])), "No accepted Source Observation baseline")


In [ ]:
PARTITION_1 = str(FIXED_TIME.date())
PARTITION_2 = str((FIXED_TIME + timedelta(days=1)).date())
PARTITION_3 = str((FIXED_TIME + timedelta(days=2)).date())
DRIFT_EXPECTED_PARTITIONS = {
    "clean": ([], [], []), "new-partition": ([PARTITION_3], [], []),
    "historical-update": ([], [PARTITION_1], []), "content-only": ([], [PARTITION_1], []), "row-deletion": ([], [PARTITION_1], []),
    "partition-deletion": ([], [], [PARTITION_1]), "restored": ([], [], []),
}

def prepare_drift_mutation(baseline_id, strategy, action, variant):
    """Construct one source mutation while preserving the accepted baseline snapshot."""
    baseline = ARTIFACTS[baseline_id]["state"]
    source, target = baseline["source"], baseline["target"]
    violation = strategy == "append" and variant in {"historical-update", "content-only", "row-deletion", "partition-deletion"}
    wanted = ("WARN" if action == "Warn" else "BLOCK") if violation else "PASS"
    data = clean_frame()
    if variant == "historical-update":
        data = make_frame({1: {"modified_at": FIXED_TIME + timedelta(seconds=1)}})
    if variant == "content-only":
        data = make_frame({1: {"name": "Updated"}})
    if variant == "row-deletion":
        data = make_frame(keep_ids=[2, 3, 4])
    if variant == "partition-deletion":
        data = make_frame(keep_ids=[3, 4])
    if variant == "new-partition":
        extra = fixture_records()[0]
        extra.update(row_id=5, business_key="K5", partition_day=FIXED_TIME.date() + timedelta(days=2))
        data = data.unionByName(spark.createDataFrame([extra], FIXTURE_SCHEMA))
    return dict(specs=[source, target], source=source, target=target, role=target["role"], frame=data,
        wanted=wanted, variant=variant, evidence=expected_rows(source, wanted, {"source_drift"}),
        previous_version=target_version(target["role"]), previous_baseline=publication_state(source, target),
        restore=lambda: put("source", clean_frame()))

def call_drift(state):
    """Observe a mutated source and execute its public Source Drift check."""
    put("source", state["frame"])
    read_physical()
    return check_source_drift(state["source"]["table_id"], target_table_id=state["target"]["table_id"],
                              spark_session=spark, verbose=False)

def verify_drift(state, artifact):
    """Verify partition classifications and unchanged publication state."""
    result = returned(artifact)
    assert_status(result, state["wanted"])
    for field, expected in zip(("new_partitions", "changed_partitions", "removed_partitions"), DRIFT_EXPECTED_PARTITIONS[state["variant"]]):
        require(sorted(map(str, result[field])) == expected, "Wrong " + field)
    require(not result["first_observation"] and not result["reappeared_partitions"], "Unexpected baseline/reappearance")
    require(target_version(state["role"]) == state["previous_version"], "Check-only call published target")
    require(publication_state(state["source"], state["target"]) == state["previous_baseline"], "Check-only call committed baseline")

for strategy in DRIFT_PROCESSING:
    for action in ("Warn", "Block"):
        baseline_id = f"drift/{strategy}/{action}/baseline"
        register(baseline_id, "drift", "Source Drift", "Real accepted baseline", "Check + Write", "BASELINE",
            lambda s=strategy, a=action: prepare_drift_baseline(s, a), call_drift_baseline, verify_drift_baseline, publication=True)
        for variant in DRIFT_EXPECTED_PARTITIONS:
            violation = strategy == "append" and variant in {"historical-update", "content-only", "row-deletion", "partition-deletion"}
            expected = ("WARN" if action == "Warn" else "BLOCK") if violation else "PASS"
            register(f"drift/{strategy}/{action}/{variant}", "drift", "Source Drift", variant, "Check", expected,
                lambda b=baseline_id, s=strategy, a=action, v=variant: prepare_drift_mutation(b, s, a, v),
                call_drift, verify_drift, publication=True, dependencies=(baseline_id,))
run_group("drift")


### 04E. Source Drift — reappearance

The reappearance target uses SCD1 so two legitimate writes can run in one Fabric activity; no activity identity is fabricated and target deletion semantics are not being tested here. A reappearance fixture first commits a real removal using an Overwrite contract, then restores the partition under the tested source strategy. This tests the persisted tombstone path separately from ordinary insertion. Setup results remain in the artifact; setup failure is BLOCKED, not a passing drift test.


In [ ]:
def prepare_reappearance(strategy, action):
    """Commit a real removal before restoring the partition under test."""
    state = prepare_drift_baseline(strategy, action, target_strategy="scd1")
    with selection(state["specs"]):
        baseline = call_drift_baseline(state)
        require(normalized(baseline["status"]) == "BASELINE", "Reappearance baseline setup failed")
    reset = seed_contract("source", [SCHEMA, FRESH, DRIFT], "Block", {"load_strategy": "overwrite"})
    with selection([reset, state["target"]]):
        put("source", make_frame(keep_ids=[3, 4]))
        read_physical()
        removal = check_source_drift(reset["table_id"], target_table_id=state["target"]["table_id"], spark_session=spark, verbose=False)
        require(normalized(removal["status"]) == "PASS", "Removal setup did not reconcile")
        publication = write_baseline(make_frame(keep_ids=[3, 4]), state["target"], [reset["table_id"]])
    wanted = ("WARN" if action == "Warn" else "BLOCK") if strategy == "append" else "PASS"
    state.update(frame=clean_frame(), wanted=wanted, setup_results=[baseline, removal, publication],
        evidence=expected_rows(state["source"], wanted, {"source_drift"}),
        previous_version=target_version(state["role"]), previous_baseline=publication_state(state["source"], state["target"]))
    return state

def verify_reappearance(state, artifact):
    """Verify tombstone-based reappearance without committing another baseline."""
    result = returned(artifact)
    assert_status(result, state["wanted"])
    require(list(map(str, result["reappeared_partitions"])) == [PARTITION_1], "Missing reappeared partition")
    require(not result["new_partitions"] and not result["changed_partitions"] and not result["removed_partitions"], "Wrong reappearance classification")
    require(target_version(state["role"]) == state["previous_version"], "Reappearance check published")
    require(publication_state(state["source"], state["target"]) == state["previous_baseline"], "Reappearance check committed baseline")

for strategy in DRIFT_PROCESSING:
    for action in ("Warn", "Block"):
        wanted = ("WARN" if action == "Warn" else "BLOCK") if strategy == "append" else "PASS"
        register(f"drift/{strategy}/{action}/reappearance", "reappearance", "Source Drift", "Reappear after accepted removal", "Check", wanted,
                 lambda s=strategy, a=action: prepare_reappearance(s, a), call_drift, verify_reappearance, publication=True)
run_group("reappearance")


## 05. Combined checks and orchestrator integration

The exhaustive matrix above calls public checks. This smaller suite checks what only integration can establish: stage order, observations, multiple simultaneous results, exception propagation, early stopping, Validate without publication and Enforce with publication.

Every Enforce scenario reserves a new publication target and freezes its contract before publishing. This avoids same-activity overwrite retry suppression. Every Enforce call has a real, explicitly selected/read source. Validate and expected Block compare both target Delta history and accepted Source Observation rows. Each preparation restores the clean physical source even when it fails.


In [ ]:
COMBINED_DQ = [
    rule("data_quality", "completeness", dict(columns=["name"], maximum_missing_percent=0, treat_blank_as_missing=True)),
    rule("data_quality", "range", dict(columns=["amount"], minimum=0, maximum=100)),
    rule("data_quality", "pattern", dict(columns=["code"], pattern="^[A-Z]{2}[0-9]{2}$")),
]
COMBINED_CHANGES = {1: {"name": None, "amount": -1.0, "code": "invalid"}}
def integration_spec(role, action):
    """Freeze integration rules while keeping source and target responsibilities explicit."""
    if role == "source":
        return seed_contract("source", [SCHEMA, FRESH, DRIFT] + COMBINED_DQ,
                             action, {"load_strategy": "append"})
    if role not in OWNED:
        reserve(role, "Silver", publication=True)
    return seed_contract(role, [SCHEMA, SENSITIVE_DEFS["mask"]] + COMBINED_DQ, action)

READ_STAGES = ["Read", "Freshness", "Schema", "Data Quality", "Profile"]
WRITE_STAGES = ["Schema", "Sensitive Data", "Source Drift", "Data Quality", "Guardrail Coverage"]

def no_publication(state):
    """Require unchanged target Delta history and accepted Source Observations."""
    require(target_version(state["target"]["role"]) == state["previous_version"], "Target Delta version changed")
    require(publication_state(state["source"], state["target"]) == state["previous_baseline"], "Accepted Source Observation changed")

def verify_stages(result, names):
    """Require the expected stage order and supported completed-stage statuses."""
    stages = result["orchestration_stages"]
    require([r["stage"] for r in stages] == names, "Incorrect orchestration stage sequence")
    require(all(r["status"] in {"passed", "warning", "skipped"} for r in stages), "Unexpected stage status")

def verify_stage_block(artifact, stage):
    """Require the intended blocking exception and no later stage execution."""
    require(isinstance(artifact["error"], RuntimeError), "Expected a real orchestration RuntimeError")
    require(f"failed during {stage}." in str(artifact["error"]), "Incorrect blocking stage")
    require(any(r["can_continue"] is False and normalized(r["status"]) == "BLOCK" for r in artifact["new_results"]), "No new blocking rule result")
    later = READ_STAGES[READ_STAGES.index(stage)+1:] if stage == "Freshness" else (
            WRITE_STAGES[WRITE_STAGES.index(stage)+1:] + ["Write", "Profile"])
    require(all(f"] {name} ... running" not in artifact["stage_output"] for name in later), "A downstream stage executed after Block")
    artifact["actual_status"] = "BLOCK"


In [ ]:
def prepare_read_integration(action, variant):
    """Construct clean, combined-violation or stale Read inputs and expectations."""
    spec = integration_spec("source", action)
    data = make_frame(COMBINED_CHANGES) if variant == "dirty" else clean_frame()
    if variant == "stale":
        data = make_frame({i: {"modified_at": datetime(1900, 1, 1)} for i in range(1, 5)})
    wanted = ("WARN" if action == "Warn" else "BLOCK") if variant in {"dirty", "stale"} else "PASS"
    evidence = []
    for doc in spec["rules"]:
        kind = doc["guardrail_type"]
        if kind == "source_drift" or variant == "stale" and kind != "freshness":
            continue
        status = wanted if kind == ("freshness" if variant == "stale" else "data_quality") else "PASS"
        evidence.append((spec, doc, status))
    return dict(specs=[spec], spec=spec, frame=data, wanted=wanted, variant=variant,
                evidence=evidence, restore=lambda: put("source", clean_frame()))

def call_read_integration(state):
    """Replace the owned source fixture and execute the real Read orchestrator."""
    put("source", state["frame"])
    return read_physical(orchestrated=True)

def verify_combined_dq(state, artifact, spec, result, dirty):
    """Apply exact DQ assertions to the nested result of an orchestration."""
    dq_spec = {**spec, "rules": [r for r in spec["rules"] if r["guardrail_type"] == "data_quality"]}
    expected = {r["rule_id"]: [1] if dirty else [] for r in dq_spec["rules"]}
    nested_state = {**state, "spec": dq_spec, "failed_ids_by_rule": expected}
    nested = {**artifact, "result": result, "error": None}
    verify_dq(nested_state, nested)

def verify_read_integration(state, artifact):
    """Verify Read completion or stopping and all applicable combined rule results."""
    if state["wanted"] == "BLOCK":
        verify_stage_block(artifact, "Freshness" if state["variant"] == "stale" else "Data Quality")
        for row in artifact["new_results"]:
            if row["guardrail_type"] == "data_quality":
                payload = json.loads(row["result_payload_json"])
                require(payload["failed_count"] == 1 and payload["failed_percent"] == 25.0, "Wrong combined blocking counts")
    else:
        result = returned(artifact)
        verify_stages(result, READ_STAGES)
        verify_combined_dq(state, artifact, state["spec"], result["dq_result"], state["variant"] == "dirty")
        artifact["actual_status"] = state["wanted"]

for action, variant in (("Block", "clean"), ("Warn", "dirty"), ("Block", "dirty"), ("Block", "stale"), ("Block", "restored")):
    expected = ("WARN" if action == "Warn" else "BLOCK") if variant in {"dirty", "stale"} else "PASS"
    register(f"integration/read/{action}/{variant}", "read-integration", "Read orchestration", variant, "Read", expected,
        lambda a=action, v=variant: prepare_read_integration(a, v), call_read_integration, verify_read_integration)
run_group("read-integration")


In [ ]:
def prepare_write_integration(action, mode, variant):
    """Read a verified source and isolate the target for the selected Write scenario."""
    role = "validation" if mode == "validate" else "publish"
    source = integration_spec("source", "Block")
    if mode == "enforce":
        role = "publish_" + uuid.uuid4().hex[:10]
        reserve(role, "Silver", publication=True)
        target = seed_contract(role, [SCHEMA, SENSITIVE_DEFS["mask"]] + COMBINED_DQ, action)
    else:
        target = integration_spec(role, action)
    data = make_frame(COMBINED_CHANGES) if variant == "dirty" else clean_frame()
    if variant == "schema-block":
        data = data.drop("secret")
    if variant == "sensitive-block":
        target = seed_contract("validation", [rule("schema", "skip"), SENSITIVE_DEFS["bucket"]])
        data = data.withColumn("age", F.col("age").cast("string"))
    wanted = ("WARN" if action == "Warn" else "BLOCK") if variant == "dirty" else (
              "BLOCK" if variant.endswith("-block") else "PASS")
    put("source", clean_frame())
    before_read = {r["guardrail_result_id"] for r in persisted_rows([source])}
    with selection([source, target]):
        source_result = read_physical(orchestrated=True)
    new_read_rows = [r for r in persisted_rows([source]) if r["guardrail_result_id"] not in before_read]
    assert_evidence(new_read_rows, expected_rows(source, "PASS", {"schema", "freshness", "data_quality"}))
    # A returned source is still checked before it can become a prerequisite.
    require(normalized(source_result["dq_result"]["status"]) == "PASS", "Source Read did not pass")
    previous_baseline = publication_state(source, target)
    drift_status = "PASS" if previous_baseline else "BASELINE"
    evidence = []
    for doc in target["rules"]:
        kind = doc["guardrail_type"]
        if variant == "schema-block" and kind != "schema" or variant == "sensitive-block" and kind == "data_quality":
            continue
        status = ("BLOCK" if variant == "schema-block" else "WARN" if variant == "sensitive-block" else "PASS") if kind == "schema" else (
                 "BLOCK" if variant == "sensitive-block" and kind == "sensitive_data" else wanted if kind == "data_quality" else "PASS")
        evidence.append((target, doc, status))
    if variant not in {"schema-block", "sensitive-block"}:
        evidence += expected_rows(source, drift_status, {"source_drift"})
    return dict(specs=[source, target], source=source, target=target, frame=data, mode=mode,
        variant=variant, wanted=wanted, evidence=evidence, source_result=source_result, source_read_results=new_read_rows,
        previous_version=target_version(target["role"]), previous_baseline=previous_baseline,
        restore=lambda: put("source", clean_frame()))

def verify_prepared_email(frame, original):
    """Require exact masked emails while preserving every other input value."""
    expected = {r["row_id"]: r.asDict() for r in original.collect()}
    values = {1: "a****************t", 2: "b***************t", 3: "a****************t", 4: "g****************t"}
    for row_id, row in expected.items():
        row["email"] = values[row_id]
    require(row_multiset(frame.collect()) == row_multiset(expected.values()), "Incorrect prepared input after Mask")

def verify_write_integration(state, artifact):
    """Verify stage behavior, prepared rows and the publication boundary."""
    if state["wanted"] == "BLOCK":
        stage = {"schema-block": "Schema", "sensitive-block": "Sensitive Data"}.get(state["variant"], "Data Quality")
        verify_stage_block(artifact, stage)
        no_publication(state)
        return
    result = returned(artifact)
    published = state["mode"] == "enforce"
    require(result.get("published") is published, "Incorrect publication decision")
    verify_stages(result, WRITE_STAGES + (["Write", "Profile"] if published else []))
    require(result["coverage_result"]["can_continue"] and not result["coverage_result"]["missing"], "Coverage did not pass")
    verify_combined_dq(state, artifact, state["target"], result["dq_result"], state["variant"] == "dirty")
    verify_prepared_email(result["dataframe"], state["frame"])
    if published:
        require(target_version(state["target"]["role"]) > state["previous_version"], "Enforce did not publish")
        require(any(json.loads(row)["observation_status"] == "drift_committed" for row in publication_state(state["source"], state["target"])), "Enforce did not commit baseline")
        actual = physical(state["target"]["role"])
        require(row_multiset(actual.select(*FIELD_NAMES).collect()) == row_multiset(result["dataframe"].collect()), "Persisted business rows differ from prepared input")
    else:
        require(result.get("validation_passed") is True, "Validate did not complete")
        no_publication(state)
    artifact["actual_status"] = state["wanted"]


In [ ]:
for mode in ("validate", "enforce"):
    for action, variant in (("Block", "clean"), ("Warn", "dirty"), ("Block", "dirty"), ("Block", "restored")):
        expected = ("WARN" if action == "Warn" else "BLOCK") if variant == "dirty" else "PASS"
        register(f"integration/write/{mode}/{action}/{variant}", "write-integration", "Write orchestration", variant,
            "Write", expected, lambda a=action, m=mode, v=variant: prepare_write_integration(a, m, v),
            lambda s: publish(s["frame"], s["target"], [s["source"]["table_id"]], mode=s["mode"]),
            verify_write_integration, publication=mode == "enforce")
for variant in ("schema-block", "sensitive-block"):
    register("integration/write/" + variant, "write-integration", "Early stopping", variant, "Write", "BLOCK",
        lambda v=variant: prepare_write_integration("Block", "validate", v),
        lambda s: publish(s["frame"], s["target"], [s["source"]["table_id"]]), verify_write_integration)
run_group("write-integration")


## 06. Coverage and contract lifecycle

Coverage is a fixed readiness gate, not an authored Warn/Block rule. It is tested after genuine public checks, including positive coverage, missing contract, missing current-version Read results, no active rules, no applicable target rules and the Development contract-free skip.

Lifecycle tests select old exact frozen versions after newer versions exist, reject missing/invalid/nonexistent/draft/foreign versions, and compare every frozen payload after execution. Invalid authoring and expression cases are explicit REJECT tests, separate from data violations.


In [ ]:
def prepare_coverage(variant):
    """Select coverage participants and declare readiness expectations."""
    source = integration_spec("source", "Block")
    target = integration_spec("validation", "Block")
    if variant == "empty-contract":
        target = seed_contract("validation", [])
    if variant == "inapplicable":
        target = seed_contract("validation", [FRESH])
    if variant == "missing-read-results":
        source = seed_contract("source", [SCHEMA, FRESH, DRIFT, rule("data_quality", "range", {
            "columns": ["amount"], "minimum": 1, "maximum": 99})], "Warn", {"load_strategy": "append"})
        # This source version's unique rules have never run through source checks.
    specs = [] if variant == "unselected" else ([target] if variant in {"missing-contract", "empty-contract", "inapplicable"} else [source, target])
    evidence = []
    if variant in {"positive", "missing-contract", "missing-read-results"}:
        evidence += expected_rows(target, "PASS")
    if variant == "positive":
        evidence += expected_rows(source, "PASS", {"freshness", "schema", "data_quality"})
    if variant in {"positive", "missing-read-results"}:
        evidence += expected_rows(source, "BASELINE", {"source_drift"})
    return dict(specs=specs, source=source, target=target, variant=variant, evidence=evidence,
                previous_version=target_version("validation"), restore=lambda: put("source", clean_frame()))

def call_coverage(state):
    """Execute genuine prerequisite checks before the public Coverage gate."""
    variant, source, target = state["variant"], state["source"], state["target"]
    if variant in {"positive", "missing-read-results"}:
        put("source", clean_frame())
        read_physical()
        if variant == "positive":
            check_freshness(source["table_id"], spark_session=spark, verbose=False)
            check_schema(clean_frame(), table_id=source["table_id"], spark_session=spark, verbose=False)
            check_dq(clean_frame(), table_id=source["table_id"], spark_session=spark, verbose=False)
    if variant in {"positive", "missing-contract", "missing-read-results"}:
        check_schema(clean_frame(), table_id=target["table_id"], spark_session=spark, verbose=False)
        sensitive = check_sensitive_data(clean_frame(), table_id=target["table_id"], spark_session=spark, verbose=False)
        check_dq(sensitive["dataframe"], table_id=target["table_id"], spark_session=spark, verbose=False)
    if variant in {"positive", "missing-read-results"}:
        check_source_drift(source["table_id"], target_table_id=target["table_id"], spark_session=spark, verbose=False)
    ids = [UNREAD["table_id"]] if variant == "missing-contract" else (
          [] if variant in {"empty-contract", "inapplicable", "unselected"} else [source["table_id"]])
    return check_guardrail_coverage(target_table_id=target["table_id"], source_table_ids=ids, spark_session=spark, verbose=False)

def verify_rejection(artifact, text=None, error_type=ValueError):
    """Require the expected exception type and optional rejection reason."""
    current, chain = artifact["error"], []
    while current is not None:
        chain.append(current)
        current = current.__cause__
    require(any(isinstance(exc, error_type) for exc in chain), f"Expected {error_type.__name__} rejection")
    if text:
        require(text.lower() in artifact["traceback"].lower(), "Rejection reason did not match: " + text)
    artifact["actual_status"] = "REJECT"

def verify_coverage(state, artifact):
    """Verify coverage readiness or rejection without publishing data."""
    variant = state["variant"]
    if variant in {"positive", "unselected"}:
        result = returned(artifact)
        assert_status(result, "SKIP" if variant == "unselected" else "PASS")
        if variant == "positive":
            require(not result["missing"] and not result["issues"] and all(c["evaluated"] for c in result["coverage"]), "Incomplete coverage")
    else:
        reason = {"missing-contract": "no selected Data Contract", "missing-read-results": "not evaluated",
                  "empty-contract": "no active Guardrail", "inapplicable": "no Guardrail applicable"}[variant]
        verify_rejection(artifact, "Pipeline is not Guardrail-ready for publication:", RuntimeError)
        require(reason in artifact["traceback"], "Wrong coverage rejection")
        artifact["actual_status"] = "BLOCK"
    require(target_version("validation") == state["previous_version"], "Coverage published business data")

for variant in ("positive", "missing-contract", "missing-read-results", "empty-contract", "inapplicable", "unselected"):
    expected = "PASS" if variant == "positive" else "SKIP" if variant == "unselected" else "BLOCK"
    register("coverage/" + variant, "coverage", "Guardrail Coverage", variant, "Check", expected,
             lambda v=variant: prepare_coverage(v), call_coverage, verify_coverage)
run_group("coverage")


### 06A. Explicit skips

These are executed tests of skip behavior. A returned SKIP is shown as the actual status; excluded or unexecuted tests are separately SKIP/BLOCKED and never PASS. `enabled=False` and no selected Development contract are tested for all five configurable public checks.


In [ ]:
def prepare_skip(kind, disabled):
    """Configure an explicit disable or an unselected Development contract."""
    source = integration_spec("source", "Block")
    target = integration_spec("validation", "Block")
    return dict(specs=[source, target], source=source, target=target, kind=kind, disabled=disabled,
                evidence=[], overrides={} if disabled else {source["table_id"]: {}, target["table_id"]: {}})

def call_skip(state):
    """Execute the public check under the selected skip configuration."""
    enabled = not state["disabled"]
    source, target, kind = state["source"], state["target"], state["kind"]
    kwargs = dict(spark_session=spark, enabled=enabled, verbose=False)
    if kind == "freshness":
        return check_freshness(source["table_id"], **kwargs)
    if kind == "source_drift":
        return check_source_drift(source["table_id"], target_table_id=target["table_id"], **kwargs)
    return check_call(kind, clean_frame(), target, enabled=enabled)

def verify_skip(state, artifact):
    """Require a skipped result without rule evaluation or data changes."""
    result = returned(artifact)
    assert_status(result, "SKIP")
    require(not result.get("checks"), "Skipped check evaluated rules")
    if "dataframe" in result:
        require(row_multiset(result["dataframe"].collect()) == row_multiset(clean_frame().collect()), "Skipped check changed data")

for kind in ("schema", "freshness", "source_drift", "data_quality", "sensitive_data"):
    for disabled in (True, False):
        label = "disabled" if disabled else "no-selected-contract"
        register(f"skip/{kind}/{label}", "skips", kind, label, "Check", "SKIP",
            lambda k=kind, d=disabled: prepare_skip(k, d), call_skip, verify_skip)
run_group("skips")


### 06B. Exact versions and rejection

These cases call real public checks with exact runtime overrides. Fractional/boolean versions are explicitly expected to be rejected rather than silently truncated to an existing version. Any runtime acceptance remains FAIL; no resolver behavior is patched.


In [ ]:
def prepare_exact_version():
    """Select an older frozen version after creating a distinct newer version."""
    old = seed_contract("checks", [SCHEMA])
    newer = seed_contract("checks", [rule("schema", "minimum_required")], "Warn")
    require(newer["version"] != old["version"], "Exact-version fixture needs distinct frozen versions")
    return dict(specs=[old], spec=old, wanted="PASS", frame=clean_frame(), evidence=expected_rows(old, "PASS"), newer=newer)

register("lifecycle/exact-old-version", "versions", "Frozen exact version", "Old frozen version after newer version exists",
    "Check", "PASS", prepare_exact_version, lambda s: check_call("schema", s["frame"], s["spec"]), verify_basic)

REJECTION_CASES = ["missing-version", "missing-id", "invalid-text", "nonexistent-version", "nonexistent-contract",
                   "draft", "foreign-table", "zero", "negative", "fractional", "boolean"]

def prepare_version_rejection(label):
    """Construct an invalid, absent, draft or foreign exact-version selection."""
    spec = seed_contract("checks", [SCHEMA])
    override = dict(contract_id=spec["contract_id"], contract_version=spec["version"])
    if label == "missing-version":
        override.pop("contract_version")
    if label == "missing-id":
        override.pop("contract_id")
    if label in {"invalid-text", "nonexistent-version", "zero", "negative", "fractional", "boolean"}:
        override["contract_version"] = {"invalid-text": "invalid", "nonexistent-version": 999999,
            "zero": 0, "negative": -1, "fractional": 1.5, "boolean": True}[label]
    if label == "nonexistent-contract":
        override["contract_id"] = str(uuid.uuid4())
    if label == "draft":
        draft = create_contract_draft(table_id=spec["table_id"], **contract_arguments())
        override = dict(contract_id=draft["contract_id"], contract_version=draft["contract_version"])
    if label == "foreign-table":
        foreign = integration_spec("source", "Block")
        override = dict(contract_id=foreign["contract_id"], contract_version=foreign["version"])
    return dict(specs=[spec], spec=spec, overrides={spec["table_id"]: override}, evidence=[])

for label in REJECTION_CASES:
    register("lifecycle/reject/" + label, "versions", "Contract rejection", label, "Check", "REJECT",
        lambda v=label: prepare_version_rejection(v), lambda s: check_call("schema", clean_frame(), s["spec"]),
        lambda s, a: verify_rejection(a))
run_group("versions")


### 06C. Invalid authoring and invalid expressions

Invalid Sensitive Data policy parameters must be rejected by the authoritative authoring service before persistence. Invalid custom expressions are evaluated through `check_dq()` with real saved/frozen contracts; rejected configuration is distinct from a Warn/Block data violation.


In [ ]:
INVALID_POLICIES = [
    ("scope", "mask", dict(scope="table", treatment="mask", preserve_start=1, preserve_end=1, mask_character="*")),
    ("negative-preserve", "mask", dict(scope="column", treatment="mask", preserve_start=-1, preserve_end=1, mask_character="*")),
    ("empty-mask", "mask", dict(scope="column", treatment="mask", preserve_start=1, preserve_end=1, mask_character="")),
    ("duplicate-bins", "bucket", dict(scope="column", treatment="bucket", bins=[0, 0], labels=["a", "b"])),
    ("unordered-bins", "bucket", dict(scope="column", treatment="bucket", bins=[10, 0], labels=["a", "b"])),
    ("label-count", "bucket", dict(scope="column", treatment="bucket", bins=[0, 10], labels=["a"])),
    ("nonnumeric-bin", "bucket", dict(scope="column", treatment="bucket", bins=[0, "ten"], labels=["a", "b"])),
    ("unknown-treatment", "unknown", dict(scope="column", treatment="unknown")),
]

def invalid_policy(treatment, parameters, action):
    """Submit invalid Sensitive Data parameters to authoritative canonicalization."""
    spec = seed_contract("checks", [SCHEMA])
    cid = next(d["column_id"] for d in COLUMN_DOCS["checks"] if d["column_name"] == "email")
    raw = dict(contract_id=spec["contract_id"], contract_version=spec["version"], table_id=spec["table_id"],
               column_id=cid, guardrail_type="sensitive_data", rule_type=treatment,
               rule_parameters_json=json.dumps(parameters), action=action)
    # Canonicalization validates an input value only; no save/freeze mutation is called.
    return canonical_guardrail_rule_record(raw, config=CONFIG, env=ENV)

for name, treatment, parameters in INVALID_POLICIES:
    for action in ("Warn", "Block"):
        register(f"authoring/reject/{name}/{action}", "invalid", "Sensitive Data authoring", name, "Authoring", "REJECT",
            lambda: {"specs": [], "evidence": []},
            lambda s, t=treatment, p=parameters, a=action: invalid_policy(t, p, a), lambda s, a: verify_rejection(a))

for name, expression in (("nonboolean", "F.col('amount')"), ("unsupported-operator", "F.col('amount') ** F.lit(2) > F.lit(0)"),
                         ("missing-column", "F.col('absent_column') > F.lit(0)")):
    definition = rule("data_quality", "custom_expression", dict(expression_language="pyspark", expression=expression))
    for action in ("Warn", "Block"):
        register(f"dq/reject/{name}/{action}", "invalid", "Custom Expression rejection", name, "Check", "REJECT",
            lambda d=definition, a=action: dict(specs=[seed_contract("checks", [d], a)], evidence=[]),
            lambda s: check_call("data_quality", clean_frame(), s["specs"][0]), lambda s, a: verify_rejection(a))
run_group("invalid")


### 06D. Frozen definition immutability

Read-only comparison of every successful freeze-service return with the final persisted row. Partial drafts remain cleanup-owned by their exact table identities. This check runs even if earlier individual tests failed.


In [ ]:
def prepare_immutability(spec):
    """Retain one exact frozen definition for a read-only comparison."""
    return dict(specs=[spec], spec=spec, evidence=[])

def read_immutable(state):
    """Read the unique persisted row for the selected frozen contract version."""
    spec = state["spec"]
    rows = metadata("METADATA_DATA_CONTRACT").where((F.col("environment_name") == ENV) &
        (F.col("contract_id") == spec["contract_id"]) & (F.col("contract_version") == spec["version"])).collect()
    require(len(rows) == 1, "Frozen contract row missing or duplicated")
    return rows[0].asDict()

def verify_immutable(state, artifact):
    """Require the frozen state and saved definition to remain unchanged."""
    result = returned(artifact)
    require(result["status"] == "frozen" and result["is_active"] is False, "Frozen lifecycle state changed")
    require(result["contract_payload_json"] == state["spec"]["record"]["contract_payload_json"], "Frozen payload changed")
    artifact["actual_status"] = "PASS"

for spec in list(CONTRACTS.values()):
    register(f"immutable/{spec['contract_id']}/{spec['version']}", "immutability", "Frozen immutability",
        "Definition unchanged after testing", "Metadata", "PASS", lambda s=spec: prepare_immutability(s), read_immutable, verify_immutable)
run_group("immutability")


## 07. Manual UI checklist

Optional reference only; widget authoring is never required to run the notebook.


<details><summary>Equivalent widget_data_contract() authoring checklist</summary>

For every item: select the test-owned table in Development, create/open a draft, select the exact column identities/types from Catalogue, author the rule, choose Warn or Block, Save and Freeze. Choose that exact frozen version in `widget_select_data_contract()` before Read/Write. Never edit a frozen row. The notebook does all of this programmatically; the UI is not a prerequisite.

- [ ] **Freshness:** choose `modified_at`, maximum age and minutes/hours/days (or lag days); choose static/no refresh for the skip case. Replace all timestamps with stale values, then restore.
- [ ] **Schema:** select Strict, Minimum Required, Relaxed or monitoring/Skip; review all frozen column types. Remove a column, change one type, or add an unexpected column as separate tests.
- [ ] **Source Drift:** set `partition_day` and `modified_at`, select Append processing; Read then Enforce Write to establish the source-target baseline. Change an already accepted partition and Read/Write again. Overwrite processing can reconcile the restoration.
- [ ] **Completeness:** select `name`, zero maximum missing percent; toggle blank-as-missing. Test null and blank separately.
- [ ] **Uniqueness:** select `business_key`, 100% minimum uniqueness; additionally select composite `category,code`. Duplicate the configured key.
- [ ] **Whitelist:** add a DQ allowed-values rule on `category` with A and B; introduce X.
- [ ] **Blacklist:** add a separate DQ blocked-values rule on `category` with DENY; introduce DENY.
- [ ] **Range:** select `amount`, 0–100 inclusive; test -1. Also author exclusive lower/upper bounds and test values exactly on those bounds.
- [ ] **Pattern:** author the format intent for `code` (two uppercase letters and two digits); review the resolved Pattern `^[A-Z]{2}[0-9]{2}$`; introduce `invalid`.
- [ ] **Column Relationship:** compare `amount` with `ceiling`; choose each supported comparison operator; adjust clean and dirty values as shown in DQ_MATRIX.
- [ ] **Conditional Completeness:** when `category=A`, require `required_note`; configure blank handling. Test `!=B` and null handling separately.
- [ ] **Conditional Values:** when `category=A`, allow `code` AA01/CC03; separately author `!=B` with a blocked ZZ99 value. Introduce ZZ99.
- [ ] **Custom Expression:** author a PySpark boolean expression `F.col('amount') >= F.lit(0)`; introduce -1.
- [ ] **Sensitive Data Mask:** select `email`, preserve one leading/trailing character, mask character `*`; inspect returned prepared DataFrame.
- [ ] **Sensitive Data Tokenize:** select `email`, Tokenize; inspect caller-owned mapping without persisting it. Repeated originals must share a token.
- [ ] **Sensitive Data Bucket:** select numeric `age`, boundaries 0/30/60 and labels young/middle/older; test exact endpoints and nulls.
- [ ] **Sensitive Data Remove:** select `secret`, Remove; verify the prepared DataFrame has no such column.
- [ ] **Sensitive failure actions:** use Schema monitoring to allow a missing governed treatment column to reach Sensitive Data; use Warn then Block frozen versions. Bucket additionally rejects nonnumeric data.
- [ ] **Guardrail Coverage:** select contracts for all participants and execute all applicable source/target checks. Omit a participant contract or switch its contract after Read to reproduce coverage failures. Coverage is a fixed Block gate, with no Warn selector.
- [ ] **Validate/Enforce:** choose Validate to run checks without publication; choose Enforce to publish after checks. Confirm target Delta history and exact contract version.

</details>


## 08. Results summary

`PASS/FAIL/SKIP/BLOCKED` is the **test verdict**, separate from the expected and actual runtime status. Expected Block/rejection is PASS only when its exact assertions succeed. An explicitly configured skip has Actual status SKIP; an excluded case has test verdict SKIP. Unexecuted/precondition-failed cases stay BLOCKED.

Investigate one case with `ARTIFACTS[test_id]`: `state`, frozen `contracts`, raw `result`, `new_results`, `error`, `traceback`, and captured `stage_output`. To rerun, use `run_case(test_id, force=True)`; if it has a baseline dependency, rerun that baseline first. Rerunning a baseline creates a fresh owned target rather than pretending a prior publication never happened. Set `TEST_FILTER` before running for a focused investigation; remember filtered prerequisites will leave dependent cases BLOCKED.

Large DQ parameter matrices can take time in Fabric because real Metadata writes and Spark actions are intentional. No time estimate or Fabric success claim is inferred from local checks.


In [ ]:
summary_df = spark.createDataFrame([tuple(str(row[c]) for c in SUMMARY_COLUMNS) for row in RESULTS.values()], SUMMARY_COLUMNS)
display(summary_df.orderBy("Test ID"))
display(summary_df.groupBy("PASS/FAIL/SKIP/BLOCKED").count())
# Declarative coverage inventory makes excluded and unexecuted groups visible.
coverage_inventory = [(group, sum(t["group"] == group for t in TASKS.values()),
    sum(t["group"] == group and RESULTS[k]["PASS/FAIL/SKIP/BLOCKED"] == "PASS" for k, t in TASKS.items()))
    for group in dict.fromkeys(t["group"] for t in TASKS.values())]
display(spark.createDataFrame(coverage_inventory, ["Group", "Registered cases", "Passing cases"]))
print("FabricOps version:", __version__, "Namespace:", RUN_NAMESPACE)
print("Registered cases:", len(TASKS), "Frozen test versions:", len(CONTRACTS))
print("Review all FAIL/BLOCKED/SKIP rows; then set cleanup confirmation to:", RUN_NAMESPACE)
FABRIC_CONTEXT.clear()
FABRIC_CONTEXT.update(BASE_CONTEXT)
builtins.FABRIC_CONTEXT = FABRIC_CONTEXT


## 09. Cleanup

Explicit opt-in **after** review. Paste the printed run namespace into the switch and run the cell. Run all leaves cleanup inert. All paths and table identities come from the notebook-local reservation manifest; metadata deletion includes Development and exact owned identities. No prefix scan, schema drop, global activity purge, activation or VACUUM is used.

Cleanup is safe after partial fixture creation and safe to retry. Do not restart the session before cleanup: the manifest is intentionally kept inside this disposable notebook. Source Observation deletion requires owned source **and** target identities. Current authoritative schemas have no `METADATA_GUARDRAIL_ROW_RESULTS`; returned failure details and token mappings are caller-owned, not invented persisted tables.


In [ ]:
CLEANUP_REVIEWED_NAMESPACE = ""  # After review, paste RUN_NAMESPACE here and run this cell explicitly.
if CLEANUP_REVIEWED_NAMESPACE != RUN_NAMESPACE:
    print("Cleanup not requested. Review section 08; then set the exact namespace in this cell.")
else:
    assert ENV == "dev" and OWNED and re.fullmatch(r"fo_gr_ut_[0-9a-f]{16}", RUN_NAMESPACE)
    ids = [s["table_id"] for s in OWNED.values()]
    assert len(ids) == len(set(ids)), "Ownership manifest has duplicate identities"
    for role, spec in OWNED.items():
        assert spec["name"] == RUN_NAMESPACE + "_" + role
        _, _, _, resolved_path = resolve_configured_lakehouse_table(
            spec["store"], spec["name"], spec["schema"], context=FABRIC_CONTEXT)
        assert resolved_path == spec["path"], "Configuration changed; refusing cleanup"
    # Validate every deletion predicate before the first mutation.
    deletions = []
    for name in META_NAMES:
        frame = metadata(name)
        columns = set(frame.columns)
        if name == "METADATA_SOURCE_OBSERVATION":
            owned_predicate = F.col("source_table_id").isin(ids) & F.col("target_table_id").isin(ids)
        else:
            assert "table_id" in columns, f"No exact ownership column for {name}; inspect before cleanup"
            owned_predicate = F.col("table_id").isin(ids)
        predicate = (F.col("environment_name") == "dev") & owned_predicate
        _, _, _, path = resolve_configured_lakehouse_table("Metadata", name,
            metadata_table_physical_schema(CONFIG, name), context=FABRIC_CONTEXT)
        deletions.append((name, path, predicate))
    for name, path, predicate in deletions:
        DeltaTable.forPath(spark, path).delete(predicate)
        assert metadata(name).where(predicate).limit(1).count() == 0, f"Cleanup incomplete: {name}"
        print("Removed exact owned rows from", name)
    for spec in OWNED.values():
        hpath = spark._jvm.org.apache.hadoop.fs.Path(spec["path"])
        fs = hpath.getFileSystem(spark._jsc.hadoopConfiguration())
        if fs.exists(hpath):
            assert fs.delete(hpath, True), f"Could not remove owned table {spec['name']}"
        assert not fs.exists(hpath)
        print("Removed owned table", spec["name"])
    for artifact in ARTIFACTS.values():
        result = artifact.get("result") or {}
        mapping = result.get("support_mapping") if isinstance(result, dict) else None
        nested_mapping = result.get("sensitive_result", {}).get("support_mapping") if isinstance(result, dict) else None
        for cached in (mapping, nested_mapping):
            if cached is not None:
                cached.unpersist()
    FABRIC_CONTEXT.clear()
    FABRIC_CONTEXT.update(BASE_CONTEXT)
    builtins.FABRIC_CONTEXT = ORIGINAL_CONTEXT if ORIGINAL_CONTEXT is not None else FABRIC_CONTEXT
    CLEANED = True
    print("Cleanup complete for", RUN_NAMESPACE, "(safe to rerun this cell).")
